## Provider Anomaly: Investigator Case Notes (v10.1, GPT-OSS)

Every referred provider gets a note that says why it was flagged: what stood out, by how much, how steady it is, what it could mean, what else could explain it, and which records would settle it. Every fact and number comes from Python; the LLM only writes sentences.

### Three layers
1. **Evidence (Python).** Features from the exploded importance file, one row per feature at its strongest analysis level, with the levels that agree, the scoring windows it appeared in (stability) and the number of model types that flagged it (consensus). Every feature has a verified dictionary row (`feature_mapping_v2_4.csv`); the run stops otherwise. Roles, holds, concern direction, provider values with peer median and percentile, main / supporting / background patterns, patient-mix detection.
2. **Schemes.** Deterministic first: a scheme is a candidate only when a main pattern's dictionary link and its measured value agree (at or beyond the 90th percentile of peers, or the 10th for low-side measures; 75 to 90 is listed as possible); evidence strength from the number of measures, their stability and the levels they come from. Then LLM validation against the evidence brief for low-evidence or diffuse cases and a 5% audit sample; the LLM can confirm or drop a candidate, never add one or raise its strength.
3. **Narrative (LLM).** For the top share of eligible providers per category (25%, floor 10) plus any provider with a high-evidence scheme. Prompt: data boundary, ranked rules, scheme definitions for the confirmed schemes only, and the brief. Every other referred provider gets a Python evidence note with the same facts.

### Outputs
- Client file: QA-passed narratives plus evidence and patient-mix notes. Quarantine file: narratives needing review or failed.
- Schemes table (one row per confirmed scheme), features file (evidence card), patient-mix list, audit file with every prompt.

### Changelog (v10.1)
- First production run (2026-10-30) tripped the circuit breaker: every narrative at 'high' reasoning effort hit the 5,400-token retry budget with 22,000+ characters of hidden analysis and no final text. Narratives now run at 'medium' ('low' for clear cases) and never 'high'; a retry after a missing or cut-off answer runs at 'low' effort with 1.8x the budget; budgets back to the v9 values (900 / 2,600 / 4,200); Layer 2 budget 900. Jobs keep provider order (no sort by effort), and the audit file records the effort used and whether the first attempt hit the limit. If the trust report shows more than 30% retries, set NARRATIVE_REASONING['default'] = 'low'.

### Changelog (v10.0)
- Merged the three-layer architecture (v9 notebook, signal_synthesis, prompt_builder v5.1) with the dictionary evidence layer of v4.2.
- From v9: scheme registry and category menus, Layer 2 validation with audit sample, temporal stability and model consensus, reasoning levels and budgets, retry factor, leak detection, circuit breaker, quarantine gate, narrative quota.
- From v4: dictionary roles and holds, concern direction, Python values and percentiles, patient mix, level-dependent handling, checkpoint and resume by prompt hash, number-grounding QA.
- New: the brief carries stability, agreement, benign explanations and records to review per main pattern; Python evidence notes for every non-narrated provider and as the client-file fallback for quarantined narratives; same-sentence number QA; fact-line grounding for dollars, percentages and percentiles; main-pattern coverage check; Layer 2 confirm-or-drop only (Layer2_Added / Layer2_Suggested / Layer2_Rejected / Layer2_Note); scheme confirmation at the 90th / 10th percentile with 75 to 90 as possible; evidence strength computed in Python; dictionary wording scrubbed of banned words before the LLM sees it; one exemplar of the note shape (placeholder until SIU-approved notes replace it).

##### Dependencies
No `--force-reinstall`: it upgrades numpy and breaks older pandas. `kernels` lets transformers load GPT-OSS in MXFP4; without it the 120B model is expanded to bf16 (about 240 GB, more than one H100). If your current v7 install cell works on this cluster, you can keep it.

In [ ]:
%pip install transformers==5.2.0 accelerate sentencepiece openai-harmony kernels
dbutils.library.restartPython()

In [ ]:
import os
import re
import json
import math
import hashlib
import unicodedata
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

from huggingface_hub import login
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

##### Configuration

In [ ]:
provider_id = 'Servicing_Provider_ID'
ROOT = 'sedo.senou@azahcccs.gov'
PROJECTS = f'/Workspace/Users/{ROOT}/Projects'
DATA_DIR = f'{PROJECTS}/all_projects_predictions/data'
EXTERNAL_DIR = f'/Workspace/Users/{ROOT}/data/external data'

with open(f'{DATA_DIR}/scoring_date.txt', 'r') as f:
    scoring_date = f.read().strip()

NOTEBOOK_VERSION = 'v10.1'
DICTIONARY_VERSION = 'v2.4'

# ── Inputs ──
IMPORTANCE_FILE = f'{DATA_DIR}/explained_top_features_{scoring_date}.csv'
# Saved by combine_all_projects_anomaly_predictions (one row per referred provider and category):
#   Servicing_Provider_ID, Category, Servicing_Provider_Anomaly_Score, *_Perspective_Score, Scoring_Date
REFERRAL_FILE = f'{DATA_DIR}/referred_providers_{scoring_date}.csv'
REFERRAL_FALLBACK_TO_RAW = True     # if that file does not exist, read the 8 referral CSVs and apply the same steps
DICTIONARY_CSV = f'{DATA_DIR}/feature_mapping_v2_4.csv'        # never the v7 file feature_mapping.csv
SCORING_FILE_TYPES = ['rolling_6m_score', 'early_score']        # peer comparisons; duplicate lines removed

# Claim category -> (project folder, referral file prefix)
CATEGORY_FOLDERS = {
    'NURSING HOME': ('nursing_home', 'nh'),
    'NEMT NOT IHS': ('nemt_not_ihs', 'nemt_not_ihs'),
    'NEMT IHS': ('nemt_ihs', 'nemt_ihs'),
    'BH PROFESSIONAL': ('bh_professional_counseling', 'bh_lpc'),
    'BH OUTPATIENT CLINIC': ('bh_outpatient_clinic', 'bh_oc'),
    'BH RESIDENTIAL TREATMENT': ('bh_residential_treatment', 'bh_rt'),
    'DME AND APPLIANCES': ('dme_and_appliances', 'dme_and_appliances'),
    'DME MEDICAL SUPPLIES': ('dme_supplies', 'dme_supplies'),
}

# ── Outputs ──
OUTPUT_DIR = DATA_DIR
CHECKPOINT_FILE = f'{OUTPUT_DIR}/llm_summary_checkpoint_{scoring_date}_{NOTEBOOK_VERSION}.jsonl'
RESUME = True                       # reuse notes written for the same prompt in an earlier run of this version and date
master_table_llm_summarization = "main.prod_results.master_anomaly_detection_referrals_and_llm_summarization"
master_table_llm_schemes = "main.prod_results.master_anomaly_detection_llm_schemes"
test_table_llm_summarization = "main.sedo.test_anomaly_detection_referrals_and_llm_summarization"

# ── Which referred providers get an LLM narrative (v9 quota: share of each category, with a floor) ──
NARRATE_SHARE = 0.25                # top share of eligible providers per category, by provider anomaly score
NARRATE_FLOOR = 10                  # at least this many per category (all of them when fewer are eligible)
NARRATE_CAP = None                  # optional maximum per category
NARRATE_HIGH_EVIDENCE = True        # a provider with a high-evidence scheme always gets a narrative

# ── Patient mix ──
DEMOGRAPHIC_CATEGORIES = {'Patient Demographics'}
TOP_K_SIGNAL = 5
PATIENT_MIX_SHARE = 0.5
PATIENT_MIX_TOP_DRIVERS = 3

# ── Pattern selection ──
MAIN_TIERS = {'top_50', 'top_80'}   # Importance_Tier values inside the 80% signal coverage
MAX_MAIN = 3
MAX_SUPPORTING = 4
MAX_BACKGROUND = 3
MAX_EVIDENCE_ITEMS = 4              # measures listed in the Python evidence note (non-narrated providers)
STRICT_COVERAGE = True              # stop the run when a feature has no dictionary row
CONFIDENCE_ORDER = ['Unverified', 'Conceptually Supported', 'Historically Supported', 'Code Verified']
MIN_CONFIDENCE_FOR_MAIN = 'Unverified'
LEVEL_PHRASE = {'provider': 'measured for the provider as a whole', 'patient': 'measured per patient, across the provider\'s patients',
                'visit': 'measured per visit', 'claim': 'measured per claim'}

# ── Temporal stability and model consensus (from the importance file; v9 rules) ──
RECENT_TIMEFRAMES = {'rolling_3m_score', 'rolling_6m_score', 'rolling_3m', 'rolling_6m'}
EARLY_TIMEFRAMES = {'early_score', 'early'}
# Wording is QA-safe on purpose: 'scoring' would trip the banned-word check ('score') in every note that repeats it
STABILITY_PHRASE = {'persistent': 'present in most review windows', 'emerging': 'present only in recent review windows',
                    'declining': 'present only in earlier review windows', 'sporadic': 'present in some review windows',
                    'unknown': ''}

# Dictionary text can carry words the notes must not use; the brief is scrubbed with these before the LLM sees it
# (fix the wording in the dictionary too: these are stop-gaps, listed in the run report when they fire)
BRIEF_WORD_SCRUB = [
    (r'\bscoring run\b', 'run'), (r'\bto score\b', 'to measure'), (r'\bscores\b', 'measures'), (r'\bscore\b', 'measure'),
    (r'\bscoring\b', 'review'), (r'\bthe model\b', 'the detection method'), (r'\bmodels\b', 'detection methods'),
    (r'\bmodel\b', 'detection method'), (r'\bbilled\b', 'paid'), (r'\binflated (mileage|miles)\b', r'\1 above the expected distance'),
    (r'\bpadded (mileage|miles|wait)\b', r'added \1'), (r'\boutliers?\b', 'unusual values'), (r'\banomal(y|ies|ous)\b', 'unusual'),
]
# Scheme words a note may use only for an allowed scheme (QA and brief filtering share this list)
SCHEME_SYNONYMS = [
    (r'double[- ]?bill|paid twice|duplicate (claims|payments)', 'duplicate billing'),
    (r'\bup-?\s?cod(e|ed|ing)\b', 'upcoding'), (r'\bphantom\b', 'phantom billing'), (r'\bunbundl', 'unbundling'),
    (r'inflated (mileage|miles)|padded (mileage|miles|wait)', 'mileage inflation'), (r'\bkickbacks?\b', 'kickback'),
]

# One SIU-approved note per category would replace this. Until then, a single exemplar shows the six-part shape; its
# numbers are not in any brief, so a model that copies them is caught by the number-grounding check.
EXEMPLAR_NOTE = (
    "Provider 123456 was flagged because its NEMT NOT IHS paid claims show far more paid claim lines per rider than peers: "
    "14.00 paid claim lines per rider against a peer median of 5.70, at the 99th percentile among 212 NEMT (non-IHS) peers. "
    "This pattern was present in most review windows and was seen at 2 analysis levels. Paid mileage per mileage line was "
    "also unusual compared with peers; no provider value is available for that measure.\n\n"
    "The pattern is consistent with excessive visits. Riders in dialysis or day programs legitimately take several round "
    "trips a week, which could account for part of the difference. In addition, the share of riders aged 65 or older was "
    "at the 90th percentile, which is context rather than a reason for the flag.\n\n"
    "Records that would clarify this: the rider roster with trip counts, the trip sheets of the ten riders with the most "
    "trips, and the medical claims paid on those trip dates.")
EXEMPLAR_MARKER = "EXAMPLE OF THE SHAPE (a different provider; never reuse its numbers or wording):\n"


# ── Python-computed values ──
MIN_PEERS = 10
MIN_CLAIMS_FOR_VALUE = 20
HARMLESS_PCT_HIGH = 50
HARMLESS_PCT_LOW = 50
SCHEME_PCT_HIGH = 90      # a linked measure confirms a scheme candidate only at or beyond this peer percentile (High direction)
SCHEME_PCT_LOW = 10       # ... or at or below this one (Low direction)
SCHEME_PCT_NEAR = 75      # between NEAR and HIGH (or 100-NEAR and LOW) the scheme is listed as possible, not confirmed

# ── Model ──
MODEL_ID = "openai/gpt-oss-120b"
# Generation budgets per reasoning effort (v9 values). First production run (2026-10-30): at 'high' the hidden analysis ran
# past 5,400 tokens on every provider and no note was produced, so narratives never use 'high' (NARRATIVE_REASONING) and a
# retry after a token-limit hit drops to 'low' effort instead of only adding tokens (RETRY_REASONING).
REASONING_TOKEN_BUDGET = {'low': 900, 'medium': 2600, 'high': 4200}
NARRATIVE_REASONING = {'clear': 'low', 'default': 'medium'}   # clear = a high-evidence scheme candidate; if the trust report shows
                                                              # many retries at 'medium', set default to 'low' (v4 ran at low)
RETRY_FACTOR = 1.8
RETRY_REASONING = 'low'
GEN_BATCH_SIZE = 1
HF_SECRET_SCOPE = None
HF_SECRET_KEY = 'hf_token'

# ── Layer 2: LLM scheme validation against the evidence brief ──
LAYER2_ENABLED = True
LAYER2_AUDIT_RATE = 0.05            # share of clear cases also validated (deterministic sample)
LAYER2_SEED = 23
LAYER2_MAX_NEW_TOKENS = 900       # Layer 2 answers are 1 to 3 lines; the budget is for the hidden analysis at 'low' effort

# ── Circuit breaker (v9) ──
BREAKER_MAX_CONSECUTIVE = 10
BREAKER_MAX_RATE = 0.5
BREAKER_MIN_SAMPLE = 20

# ── Scheme registry: plain paid-claims wording; no intent, no medical-necessity judgment ──
SCHEME_REGISTRY = {
    'upcoding': 'payment for a more expensive code than peers are paid for with similar patients, or for a diagnosis rarely paired with that equipment or service among peers. Upcoding is about which code or pairing is paid; service upcreep is a drift of the code mix over time; level of care upcoding is the facility level-of-care tier',
    'unbundling': 'payment for several line items or claims where peers are paid for one combined service',
    'modifier misuse': 'payment with modifiers that change reimbursement (for example RR = rental) in combinations peers rarely use',
    'diagnosis inflation/padding': 'unspecified or rarely seen diagnoses on paid claims far more often than at peers',
    'excessive visits': 'far more paid encounters per patient than peers have for the same type of service. Excessive visits is the encounter count; frequency abuse is the interval or quantity against a limit; impossible day billing is what fits in one day',
    'frequency abuse': 'paid units or services above a policy frequency or quantity limit, or far above peers for the same code',
    'phantom billing': 'paid services with no supporting activity in the claims data, such as trips with no medical claim for the member that day; this is a gap to check, not proof the service did not happen',
    'duplicate billing': 'the same service paid more than once for the same patient and date',
    'service upcreep': 'a shift toward higher-cost codes over time, faster than peers serving similar patients',
    'place-of-service manipulation': 'payment with a place of service that peers rarely pair with the procedure',
    'billing for non-covered services': "payment for services outside the member's covered benefit or eligibility, or for non-covered trip purposes; weekend or holiday dates are a timing pattern, not non-covered",
    'high-cost outlier manipulation': 'payments concentrated in a few high-cost codes or claims far above peers (payment size, not code choice or volume)',
    'impossible day billing': 'time-based units for one patient on one day that add up to more hours than a day holds',
    'rental vs purchase manipulation': 'rental and purchase payments for the same equipment in patterns peers rarely show',
    'mileage inflation': 'paid transport miles or wait-time units far above what peers are paid for similar trips',
    'geographic improbability': 'trip distances or patient locations that are implausible for the service area',
    'level of care upcoding': 'payment at a higher level of care than peers use for similar patients',
    'group therapy upcoded to individual': 'individual therapy codes paid where peers mostly use group codes',
    'medically unnecessary services': 'a billing-pattern label only: volume, patient-panel or payment concentration that investigators review for medical necessity; the claims data cannot assess necessity',
    'medically unnecessary dme': 'a billing-pattern label only: equipment payments whose diagnosis pairings or volume differ from peers; the claims data cannot assess necessity',
    'ineligible hospice enrollment': 'payment patterns consistent with hospice benefits outside eligibility windows',
    'other': 'a paid-claims pattern that matches no defined scheme',
}
KNOWN_SCHEMES = list(SCHEME_REGISTRY)
SCHEME_DEFINITIONS_BY_LABEL = {k: v for k, v in SCHEME_REGISTRY.items() if k != 'other'}
NEW_PATTERN_LABEL = 'No known scheme (possible new pattern)'
# A single measure cannot support these labels: confirmed only when two different main patterns link to them
SCHEMES_NEEDING_TWO_MEASURES = {'place-of-service manipulation', 'phantom billing', 'duplicate billing',
                                'billing for non-covered services'}
# Schemes Layer 2 may discuss per category (v9 menus); the dictionary links of the main patterns are always added.
# 'medically unnecessary services/dme' stay in the registry for definitions but are off every menu: necessity cannot be judged from paid claims.
CATEGORY_SCHEME_MENU = {
    'NURSING HOME': ['upcoding', 'level of care upcoding', 'unbundling', 'high-cost outlier manipulation', 'duplicate billing',
                     'phantom billing'],
    'BH RESIDENTIAL TREATMENT': ['level of care upcoding', 'impossible day billing', 'excessive visits', 'unbundling', 'upcoding',
                                 'diagnosis inflation/padding', 'phantom billing'],
    'BH OUTPATIENT CLINIC': ['unbundling', 'upcoding', 'excessive visits', 'group therapy upcoded to individual',
                             'impossible day billing', 'place-of-service manipulation', 'phantom billing'],
    'BH PROFESSIONAL': ['excessive visits', 'frequency abuse', 'upcoding', 'group therapy upcoded to individual',
                        'impossible day billing', 'phantom billing'],
    'NEMT NOT IHS': ['mileage inflation', 'geographic improbability', 'phantom billing', 'upcoding', 'place-of-service manipulation',
                     'billing for non-covered services', 'duplicate billing', 'modifier misuse'],
    'NEMT IHS': ['mileage inflation', 'geographic improbability', 'phantom billing', 'upcoding', 'place-of-service manipulation',
                 'billing for non-covered services', 'duplicate billing', 'modifier misuse'],
    'DME AND APPLIANCES': ['upcoding', 'modifier misuse', 'rental vs purchase manipulation',
                           'unbundling', 'service upcreep', 'phantom billing', 'high-cost outlier manipulation'],
    'DME MEDICAL SUPPLIES': ['upcoding', 'modifier misuse', 'unbundling', 'frequency abuse',
                             'service upcreep', 'phantom billing'],
}
POPULATION_CONTEXT = {
    'NURSING HOME': 'Medicaid nursing facility residents', 'BH RESIDENTIAL TREATMENT': 'Medicaid members in behavioral health residential treatment',
    'BH OUTPATIENT CLINIC': 'Medicaid members receiving outpatient behavioral health services',
    'BH PROFESSIONAL': 'Medicaid members receiving professional counseling services',
    'NEMT NOT IHS': 'Medicaid members using non-emergency medical transportation (non-IHS providers)',
    'NEMT IHS': 'Medicaid members using non-emergency medical transportation (IHS and tribal providers)',
    'DME AND APPLIANCES': 'Medicaid members receiving durable medical equipment and appliances',
    'DME MEDICAL SUPPLIES': 'Medicaid members receiving medical supplies',
}
CATEGORY_DISPLAY = {
    'NEMT NOT IHS': 'NEMT (non-IHS)', 'NEMT IHS': 'NEMT (IHS)', 'DME MEDICAL SUPPLIES': 'DME medical supplies',
    'DME AND APPLIANCES': 'DME and appliances', 'NURSING HOME': 'nursing home',
    'BH OUTPATIENT CLINIC': 'BH outpatient clinic', 'BH RESIDENTIAL TREATMENT': 'BH residential treatment',
    'BH PROFESSIONAL': 'BH professional'}
CATEGORY_ALIASES = {
    'bh outpatient clinic': ['behavioral health outpatient clinic', 'behavioral health outpatient', 'bh outpatient'],
    'bh residential treatment': ['behavioral health residential treatment', 'behavioral health residential', 'bh residential'],
    'bh professional': ['behavioral health professional', 'bh professional counseling', 'behavioral health counseling'],
    'nemt not ihs': ['nemt (not ihs)', 'nemt (non-ihs)', 'non-ihs nemt', 'nemt non-ihs',
                     'non-emergency medical transportation (not ihs)', 'non-emergency medical transportation not ihs'],
    'nemt ihs': ['nemt (ihs)', 'ihs nemt', 'non-emergency medical transportation ihs'],
    'dme and appliances': ['durable medical equipment and appliances', 'dme appliances'],
    'dme medical supplies': ['durable medical equipment medical supplies', 'durable medical equipment supplies', 'dme supplies'],
    'nursing home': ['nursing facility', 'nursing home care'],
}
print(f"{NOTEBOOK_VERSION} | dictionary {DICTIONARY_VERSION} | scoring date {scoring_date}")

##### Helpers

In [ ]:
# ── Text and number helpers ──
# GPT-OSS writes narrow no-break spaces, non-breaking hyphens and curly quotes; every check runs on normalized text.
_UNI_MAP = str.maketrans({
    **{chr(c): '-' for c in (0x2010, 0x2011, 0x2012, 0x2013, 0x2014, 0x2015, 0x2212)},
    **{chr(c): ' ' for c in (0x00a0, 0x2007, 0x2009, 0x202f)},
    chr(0x200b): '', chr(0x2018): "'", chr(0x2019): "'", chr(0x201c): '"', chr(0x201d): '"', chr(0x00d7): 'x',
})
_DIGIT_GROUP_SPACE = re.compile('(?<=\\d)[' + chr(0x00a0) + chr(0x2007) + chr(0x2009) + chr(0x202f) + '](?=\\d{3}(?!\\d))')


def normalize_text(s):
    if s is None:
        return ''
    s = _DIGIT_GROUP_SPACE.sub(',', str(s))
    return unicodedata.normalize('NFKC', s).translate(_UNI_MAP)


def normalize_provider_id(series):
    """Provider IDs as clean strings ('1029', not 1029.0); blanks become NaN."""
    s = series.astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return s.where(~s.isin(['', 'nan', 'None', 'NaN', '<NA>']))


def normalize_category(series):
    return series.astype(str).str.strip().str.upper()


def same_date(series, date_str):
    """True where a date column equals date_str, whatever its format ('2026-10-12', '2026-10-12 00:00:00', ...)."""
    want = pd.to_datetime(date_str).strftime('%Y-%m-%d')
    s = series.astype(str).str.strip()
    parsed = {v: pd.to_datetime(v, errors='coerce') for v in s.unique()}     # per value: mixed formats are fine
    return s.map(lambda v: (not pd.isna(parsed[v])) and parsed[v].strftime('%Y-%m-%d') == want).astype(bool)


def _norm_words(s):
    return ' ' + re.sub(r'[^a-z0-9]+', ' ', normalize_text(s).lower()).strip() + ' '


def category_mentioned(text, category):
    """True if the claim category (or an alias) appears as whole words; longer names of other categories are removed
    first so 'durable medical equipment supplies' does not count as DME AND APPLIANCES."""
    s = _norm_words(text)
    cat = str(category).lower().strip()
    own = [_norm_words(a) for a in [cat] + CATEGORY_ALIASES.get(cat, [])]
    for other, aliases in CATEGORY_ALIASES.items():
        if other == cat:
            continue
        for a in sorted([other] + aliases, key=len, reverse=True):
            na = _norm_words(a)
            if any(o.strip() in na.strip() for o in own) and na not in own:
                s = s.replace(na, ' ')
    if cat == 'nemt ihs' and re.search(r' no[nt] ihs ', s):
        return False
    return any(o in s for o in own)


def ordinal(n):
    n = int(round(float(n)))
    suf = 'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f"{n}{suf}"


def pct_ordinal(p):
    """Percentile as an ordinal between 1st and 99th."""
    return ordinal(min(max(float(p), 1), 99))


def percentile_of(values, v):
    """Percent of peers below v, ties counting half. values include v."""
    vals = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if vals.empty or v is None or pd.isna(v):
        return np.nan
    return float(((vals < v).mean() + 0.5 * (vals == v).mean()) * 100)


def fmt_value(x, vtype):
    if x is None or (not isinstance(x, str) and pd.isna(x)):
        return None
    if isinstance(x, str):
        return x
    x = float(x)
    if vtype == 'Dollars':
        return f"${x:,.0f}" if abs(x) >= 100 else f"${x:,.2f}"
    if vtype in ('Share (0-1)', 'Category level', 'Tier label', 'Yes/No flag'):
        if not 0 <= x <= 1:
            return f"{x:,.1f}"
        x = 0.999 if 0.999 <= x < 1 else (0.001 if 0 < x < 0.001 else x)
        return f"{x:.1%}"
    if vtype in ('Count', 'Days', 'Miles', 'Units'):
        return f"{x:,.0f}" if float(x).is_integer() else f"{x:,.1f}"
    return f"{x:,.2f}"

##### Code reference tables
Procedure, diagnosis and modifier descriptions for codes inside feature names.

In [ ]:
# ── Code reference tables (procedure, diagnosis, modifier descriptions) ──
icd10_desc = pd.read_csv(f'{EXTERNAL_DIR}/icd10_categories_description.csv')
hcpcs_desc = pd.read_csv(f'{EXTERNAL_DIR}/hcpcs_code_description.csv')
cpt_desc = pd.read_csv(f'{EXTERNAL_DIR}/cpt_codes_description.csv')
modifier_desc = pd.read_csv(f'{EXTERNAL_DIR}/procedure_modifiers.csv')
diag_hierarchy = pd.read_csv(f'{EXTERNAL_DIR}/icd_10_diagnosis_code_root_and_category.csv', dtype=str)


def _build_lookup(df, candidate_pairs):
    for code_col, desc_col in candidate_pairs:
        if code_col in df.columns and desc_col in df.columns:
            d = df.dropna(subset=[code_col, desc_col])
            return dict(zip(d[code_col].astype(str).str.strip().str.upper(), d[desc_col].astype(str).str.strip()))
    cols = df.columns.tolist()
    if len(cols) >= 2:
        d = df.dropna(subset=cols[:2])
        return dict(zip(d[cols[0]].astype(str).str.strip().str.upper(), d[cols[1]].astype(str).str.strip()))
    return {}


proc_code_lookup = _build_lookup(hcpcs_desc, [('HCPCS Codes', 'Description'), ('hcpcs_code', 'description'),
                                              ('Code', 'Description'), ('HCPCS_CD', 'Description')])
for k, v in _build_lookup(cpt_desc, [('CPT Code', 'Description'), ('cpt_code', 'description'),
                                     ('Code', 'Description'), ('CPT_CD', 'Description')]).items():
    proc_code_lookup.setdefault(k, v)
diag_code_lookup = {k.replace('.', ''): v for k, v in _build_lookup(icd10_desc, [
    ('ICD10 Code', 'Description'), ('icd10_code', 'description'), ('Code', 'Description'),
    ('DIAG_CD', 'Description'), ('ICD-10-CM Code', 'Description')]).items()}
modifier_lookup = _build_lookup(modifier_desc, [('Modifier', 'Description'), ('modifier', 'description'),
                                                ('Code', 'Description'), ('PROC_MOD', 'Description'),
                                                ('Modifier Code', 'Modifier Description')])
_dh = diag_hierarchy.rename(columns=lambda c: c.strip()).rename(columns={'diag_cd_root': 'diag_root'})
_dh['diag_root'] = _dh['diag_root'].astype(str).str.strip().str.upper()
if _dh['diag_root'].duplicated().any():
    print(f"WARNING: {int(_dh['diag_root'].duplicated().sum())} duplicate roots in the ICD-10 group file (first kept).")
_dh = _dh.drop_duplicates('diag_root')
diag_group_lookup = {k: v for k, v in zip(_dh['diag_root'], _dh['category_level_III'].fillna('').astype(str).str.strip()) if v}
print(f"Codes described: procedures {len(proc_code_lookup):,} | diagnoses {len(diag_code_lookup):,} | "
      f"diagnosis groups {len(diag_group_lookup):,} | modifiers {len(modifier_lookup):,}")

CODES_WITHOUT_DESCRIPTION = Counter()
CODES_DESCRIBED_BY_GROUP = Counter()


def _cap(s):
    s = str(s).strip()
    return s[:1].upper() + s[1:] if s else s


def describe_diag(code):
    c = str(code).strip().upper().replace('.', '')
    if not c:
        return ''
    if c in diag_code_lookup:
        return f"Diagnosis {c}: {_cap(diag_code_lookup[c])}"
    if c[:3] in diag_group_lookup:
        CODES_DESCRIBED_BY_GROUP[c] += 1
        return f"Diagnosis {c} (group {c[:3]}: {_cap(diag_group_lookup[c[:3]])})"
    CODES_WITHOUT_DESCRIPTION[c] += 1
    return f"Diagnosis {c}"


def describe_proc(code):
    c = str(code).strip().upper()
    if c in ('', '0', '-1'):
        return ''
    if c in proc_code_lookup:
        return f"Procedure {c}: {_cap(proc_code_lookup[c])}"
    CODES_WITHOUT_DESCRIPTION[c] += 1
    return f"Procedure {c}"


def describe_mods(mod_part):
    out = []
    for m in [m.strip().upper() for m in str(mod_part).split(',') if m.strip() not in ('-1', '0', '')]:
        if m in modifier_lookup:
            out.append(f"{m}={_cap(modifier_lookup[m])}")
        else:
            CODES_WITHOUT_DESCRIPTION[m] += 1
            out.append(m)
    return f"Modifiers: {', '.join(out)}" if out else ''


def decode_feature_codes(feature):
    """Codes embedded in a feature name, with their descriptions: 'Procedure E1390: ... | Modifiers: RR=Rental'."""
    s = str(feature)
    parts = []
    for prefix in ('procmodif_proc_pair_', 'proc_proc_mod_other_'):
        if s.startswith(prefix) and '__' in s:
            mods, proc = s[len(prefix):].split('__', 1)
            parts = [describe_proc(proc), describe_mods(mods)]
            break
    else:
        m = re.match(r'^(?:diag_cd_\d_proc_cd_pair_|diag_proc_other_)(.+?)__(.+)$', s)
        if m:
            parts = [describe_diag(m.group(1)), describe_proc(m.group(2))]
        elif s.startswith('PROC_CD_'):
            parts = [describe_proc(s[len('PROC_CD_'):])]
        elif re.match(r'^PROC_MOD_\d_', s):
            parts = [describe_mods(s.split('_', 3)[3])]
    return ' | '.join(p for p in parts if p)

##### Feature dictionary (v2.4)
One row per model feature root; one-hot levels (`POS_DESC_HOME`) and count levels (`line_item_cnt_42` -> `line_item_cnt_N`) map to their root.

In [ ]:
# ── Feature dictionary v2.2 ──
mapping_df = pd.read_csv(DICTIONARY_CSV, dtype=str).fillna('')
REQUIRED_MAPPING_COLS = ['Feature_Root', 'Replacement', 'Description', 'Suspicious_Pattern', 'Benign_Explanations',
                         'Narrative_Caveat', 'Value_Template', 'Records_To_Review', 'Policy_Lever', 'Narrate_Role',
                         'Validation_Status', 'Applies_To', 'Concept_Group', 'Measured_At', 'Value_Type',
                         'Scheme_Link', 'Concern_Direction']
_missing_cols = [c for c in REQUIRED_MAPPING_COLS if c not in mapping_df.columns]
assert not _missing_cols, f"{DICTIONARY_CSV} is not a v2.2 export; missing columns: {_missing_cols}"
for c in ('Definition_Confidence', 'Level_Dependent'):          # v2.3 columns; older exports lack them
    if c not in mapping_df.columns:
        mapping_df[c] = 'Unverified' if c == 'Definition_Confidence' else ''
for c in mapping_df.columns:
    mapping_df[c] = mapping_df[c].astype(str).str.strip()
mapping_df.loc[~mapping_df['Definition_Confidence'].isin(CONFIDENCE_ORDER), 'Definition_Confidence'] = 'Unverified'
assert mapping_df['Feature_Root'].is_unique, "Duplicate Feature_Root rows in the dictionary"
assert (mapping_df['Description'] != '').all(), "Every dictionary row must have a Description"
assert set(mapping_df['Narrate_Role']) <= {'Driver', 'Context only', 'Do not narrate'}, set(mapping_df['Narrate_Role'])
assert set(mapping_df['Concern_Direction']) <= {'High', 'Low', 'Either', ''}, set(mapping_df['Concern_Direction'])
_bad_links = {s.strip() for v in mapping_df['Scheme_Link'] for s in v.split('|')
              if s.strip() and s.strip().lower() != 'none' and s.strip().lower() not in KNOWN_SCHEMES}
assert not _bad_links, f"Scheme_Link labels not in KNOWN_SCHEMES: {_bad_links}"

MAP = mapping_df.set_index('Feature_Root').to_dict('index')
ONE_HOT_VALUE_TYPES = {'Category level', 'Tier label', 'Yes/No flag', 'Date part'}
SUFFIX_OK_ROOTS = sorted((r for r, v in MAP.items() if v['Value_Type'] in ONE_HOT_VALUE_TYPES), key=len, reverse=True)
COUNT_LEVEL_ROOTS = {r for r, v in MAP.items() if v['Value_Type'] == 'Count level'}     # e.g. line_item_cnt_N


def match_root(feature):
    """Dictionary row of a model feature: exact name; one-hot root + '_level'; or count level 'name_42' -> 'name_N'."""
    s = str(feature).strip()
    if s in MAP:
        return s
    for r in SUFFIX_OK_ROOTS:
        if s.startswith(r + '_'):
            return r
    m = re.match(r'^(.+)_(\d+)$', s)
    if m and f'{m.group(1)}_N' in COUNT_LEVEL_ROOTS:
        return f'{m.group(1)}_N'
    return ''


# ── Levels carried in one-hot feature names, in plain words ──
TIER_ROOTS = {'average_procedure_zscore_class', 'global_procedure_zscore_class', 'mileage_zscore_class',
              'not_mileage_zscore_class', 'procedure_volume_zscore_class', 'reservation_zscore_class',
              'wait_time_zscore_class'}
TIER_PLAIN = {   # z-scores are capped at 10 upstream, so 'Rare Anomaly' is the highest tier reached
    'extreme low underuse': 'far below peers', 'very low underuse': 'well below peers',
    'slightly below average underuse': 'below peers', 'slightly below avg underuse': 'below peers',
    'normal': 'in line with peers, or too little data to compare',
    'mild anomaly': 'somewhat above peers', 'anomaly': 'well above peers',
    'rare anomaly': 'far above peers (highest level)', 'extreme anomaly': 'far above peers (highest level)',
    'insanely rare': 'far above peers (highest level)',
}
WAIT_BANDS = {   # unusual_wait_time_class: fixed hours (1 T2007 unit = 30 minutes)
    'normal': 'up to 2 hours', 'high': '2 to 4 hours', 'extremely high': '4 to 8 hours',
    'abnormal': '8 to 24 hours', 'suspicious': 'more than 24 hours', 'no wait time': 'no wait time',
}
MULTI_DIM_PLAIN = {
    'normal': 'no measure unusual', 'single dimension': 'unusual on one measure',
    'multi dimension moderate': 'unusual on several measures', 'multi dimension high': 'unusual on many measures',
    'multi dimension extreme': 'unusual on most measures',
}
# Levels saying the provider is normal or below peers, or a flag is absent: background only, never a reason
NON_CONCERNING_LEVELS = {'in line with peers, or too little data to compare', 'below peers', 'well below peers',
                         'far below peers', 'not present', 'up to 2 hours', 'no wait time',
                         'below the top 5% of providers', 'no measure unusual'}
# Levels that only say a value was missing: never narrated
MISSING_LEVEL_WORDS = {'', 'unknown', 'null', 'nan', 'none', 'missing', 'other unknown', 'not available', 'n/a'}
RISKY_LEVEL_WORDS = re.compile(r'anomal|suspicious|rare|outlier|extreme|insane|abnormal|watch', re.I)
UNKNOWN_LEVELS = set()


def raw_level(feature, root):
    s = str(feature).strip()
    if root in COUNT_LEVEL_ROOTS:
        return s.rsplit('_', 1)[1]
    return s[len(root) + 1:] if len(s) > len(root) else ''


def _clean_code_level(raw):
    """'RR,N1,-1,-1__E1390' -> 'RR, N1 with E1390' (placeholders -1 and 0 removed); '' when only placeholders."""
    out = []
    for part in raw.split('__'):
        toks = [t.strip() for t in part.split(',') if t.strip() not in ('', '-1', '0')]
        if toks:
            out.append(', '.join(toks))
    return ' with '.join(out)


def plain_level(feature, root):
    """(level text, is_missing) for a one-hot or count-level feature; ('', False) for a plain feature."""
    raw = raw_level(feature, root)
    if not raw:
        return '', False
    key = raw.replace('_', ' ').strip().lower()
    vtype = MAP[root]['Value_Type']
    if root in COUNT_LEVEL_ROOTS:
        return raw, False
    if root in TIER_ROOTS:
        if key in TIER_PLAIN:
            return TIER_PLAIN[key], False
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level', False
    if root == 'unusual_wait_time_class':
        if key in WAIT_BANDS:
            return WAIT_BANDS[key], False
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual wait-time band', False
    if root == 'multi_dim_anomaly_class':
        if key in MULTI_DIM_PLAIN:
            return MULTI_DIM_PLAIN[key], False
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level', False
    if root in ('visit_frequency_class', 'weeks_with_unusual_visits_class'):
        m = re.search(r'above\s*(\d+)', key)
        if m:
            return f"top {100 - int(m.group(1))}% of providers", False
        if 'below' in key:
            return 'below the top 5% of providers', False
    if vtype == 'Yes/No flag':
        if key in ('1', '1.0', 'true', 'y', 'yes'):
            return '', False
        if key in ('0', '0.0', 'false', 'n', 'no'):
            return 'not present', False
    if vtype == 'Category level' and ('__' in raw or ',' in raw):
        cleaned = _clean_code_level(raw)
        return cleaned, cleaned == ''
    if key in MISSING_LEVEL_WORDS:
        return raw, True
    if vtype == 'Tier label' and RISKY_LEVEL_WORDS.search(key):
        UNKNOWN_LEVELS.add((root, raw))
        return 'an unusual level', False
    return raw.replace('_', ' ').strip(), False


def feature_label(root, level):
    label = MAP[root]['Replacement']
    if '{level}' in label:
        return label.replace('{level}', level)
    return f"{label}: {level}" if level else label


def applies_to(applies, category):
    a, cat = str(applies).strip().upper(), str(category).strip().upper()
    if a in ('', 'ALL CATEGORIES'):
        return True
    if a.startswith('ALL EXCEPT'):
        return cat not in [x.strip() for x in a.replace('ALL EXCEPT', '').split(';')]
    return cat in [x.strip() for x in a.split(';')]


print(f"Dictionary {DICTIONARY_VERSION}: {len(MAP)} rows | Driver {sum(v['Narrate_Role'] == 'Driver' for v in MAP.values())} | "
      f"Context only {sum(v['Narrate_Role'] == 'Context only' for v in MAP.values())} | on hold "
      f"{sum(v['Validation_Status'] == 'Hold until confirmed' for v in MAP.values())} | one-hot roots {len(SUFFIX_OK_ROOTS)} | "
      f"count-level rows {len(COUNT_LEVEL_ROOTS)}")

##### Load the exploded importance file

In [ ]:
# ── Exploded importance file: one row per provider x Analysis_Level x feature ──
IMPORTANCE_COLS = [provider_id, 'Category', 'Analysis_Level', 'Importance_Tier', 'Analysis_Levels_In_Agreement',
                   'Which_Analysis_Levels', 'Feature_Name', 'Feature_Category', 'Feature_Rank_The_Provider',
                   'Overall_Importance_Score', 'Share_Of_Provider_Signal', 'Cumulative_Signal_Coverage', 'Scoring_Date']
# Present in the consolidated explainability output; used for stability and consensus, defaulted when absent
OPTIONAL_COLS = ['Which_Timeframe_Flagged', 'Timeframes_Detected_In', 'Models_Detected_In', 'Scoring_Runs_Detected_In']
_imp_raw = pd.read_csv(IMPORTANCE_FILE, dtype={provider_id: str}, low_memory=False)
_missing = [c for c in IMPORTANCE_COLS if c not in _imp_raw.columns]
if _missing:
    raise ValueError(f"{IMPORTANCE_FILE} is not the exploded explained_top_features file; missing {_missing}")
for c in OPTIONAL_COLS:
    if c not in _imp_raw.columns:
        print(f"NOTE: {c} not in the importance file; stability or consensus will read 'unknown'")
        _imp_raw[c] = np.nan
imp = _imp_raw[IMPORTANCE_COLS + OPTIONAL_COLS].copy()
imp[provider_id] = normalize_provider_id(imp[provider_id])
imp['Category'] = normalize_category(imp['Category'])
imp['Feature'] = imp['Feature_Name'].astype(str).str.strip()
imp['Feature_Category'] = imp['Feature_Category'].fillna('').astype(str).str.strip()
imp['Importance_Tier'] = imp['Importance_Tier'].fillna('').astype(str).str.strip()
imp['Which_Timeframe_Flagged'] = imp['Which_Timeframe_Flagged'].fillna('').astype(str)
for c in ['Feature_Rank_The_Provider', 'Overall_Importance_Score', 'Analysis_Levels_In_Agreement',
          'Timeframes_Detected_In', 'Models_Detected_In', 'Scoring_Runs_Detected_In']:
    imp[c] = pd.to_numeric(imp[c], errors='coerce')

load_stats = Counter(importance_rows=len(imp))
_date_ok = same_date(imp['Scoring_Date'], scoring_date)
load_stats['dropped_other_scoring_date'] = int((~_date_ok).sum())
imp = imp[_date_ok]
if imp.empty:
    raise ValueError(f"No rows for scoring date {scoring_date} in {IMPORTANCE_FILE}; dates in file: "
                     f"{sorted(_imp_raw['Scoring_Date'].astype(str).unique())[:5]}")
_bad = imp[provider_id].isna() | imp['Overall_Importance_Score'].isna() | imp['Feature_Rank_The_Provider'].isna()
load_stats['dropped_blank_id_score_or_rank'] = int(_bad.sum())
imp = imp[~_bad]
_idx = imp['Feature'].str.lower() == 'index'
load_stats['dropped_index_feature'] = int(_idx.sum())
imp = imp[~_idx].copy()


# ── Temporal stability and model consensus (v9 rules), per row ──
def temporal_stability(which_tf, n_tf):
    """persistent: 3+ scoring windows; emerging: recent windows only; declining: early only; sporadic: otherwise."""
    tfs = {t.strip().lower() for t in str(which_tf or '').split(',') if t.strip()}
    if not tfs:
        return 'unknown'
    n = max(len(tfs), 0 if pd.isna(n_tf) else int(n_tf))
    if n >= 3:
        return 'persistent'
    has_recent, has_early = bool(tfs & RECENT_TIMEFRAMES), bool(tfs & EARLY_TIMEFRAMES)
    if has_recent and not has_early:
        return 'emerging'
    if has_early and not has_recent:
        return 'declining'
    return 'sporadic'


def model_consensus(n_models):
    """How many model types flagged the feature: unanimous (4+), strong (3), partial (2), weak (1 or unknown)."""
    if pd.isna(n_models):
        return 'unknown'
    return 'unanimous' if n_models >= 4 else 'strong' if n_models >= 3 else 'partial' if n_models >= 2 else 'weak'


imp['Temporal_Stability'] = [temporal_stability(w, n) for w, n in zip(imp['Which_Timeframe_Flagged'], imp['Timeframes_Detected_In'])]
imp['Model_Consensus'] = imp['Models_Detected_In'].map(model_consensus)

_unknown_tiers = sorted(set(imp['Importance_Tier']) - MAIN_TIERS)
print(f"Importance rows: {len(imp):,} | providers {imp[provider_id].nunique()} | "
      f"levels {imp['Analysis_Level'].value_counts().to_dict()}")
print(f"Importance_Tier values: {imp['Importance_Tier'].value_counts().to_dict()} "
      f"(main patterns come only from {sorted(MAIN_TIERS)})")
print(f"Stability: {imp['Temporal_Stability'].value_counts().to_dict()} | consensus: {imp['Model_Consensus'].value_counts().to_dict()}")
print("Load:", dict(load_stats))

##### Load referred providers

In [ ]:
# ── Referred providers and their anomaly score ──
REFERRAL_COLS = [provider_id, 'Category', 'Servicing_Provider_Anomaly_Score', 'Provider_Perspective_Score',
                 'Patient_Perspective_Score', 'Visit_Perspective_Score', 'Claim_Perspective_Score']

if os.path.exists(REFERRAL_FILE):
    referrals = pd.read_csv(REFERRAL_FILE, dtype={provider_id: str})
    REFERRAL_SOURCE = REFERRAL_FILE
    if 'Scoring_Date' in referrals.columns:
        _ok = same_date(referrals['Scoring_Date'], scoring_date)
        if not _ok.all():
            print(f"WARNING: dropping {int((~_ok).sum())} referral rows with another Scoring_Date")
        referrals = referrals[_ok]
elif REFERRAL_FALLBACK_TO_RAW:
    # Same steps as combine_all_projects_anomaly_predictions (rename, category names, BH PROFESSIONAL payout filter).
    # Keep in sync, or save referred_providers_{scoring_date}.csv from that notebook instead.
    print(f"WARNING: {REFERRAL_FILE} not found; reading the 8 referral files directly.")
    _raw_cat = {prefix: cat for cat, (_, prefix) in CATEGORY_FOLDERS.items()}
    _parts = []
    for cat, (folder, prefix) in CATEGORY_FOLDERS.items():
        p = f'{PROJECTS}/{folder}/referrals/{prefix}_monthly_anomaly_detection_referrals.csv'
        if os.path.exists(p):
            _parts.append(pd.read_csv(p, dtype={'SProv_ID': str}))
        else:
            print(f"  missing referral file for {cat}: {p}")
    referrals = pd.concat(_parts, ignore_index=True)
    referrals['category'] = referrals['category'].replace(_raw_cat)
    referrals = referrals.rename(columns={
        'category': 'Category', 'SProv_ID': provider_id, 'monthly_payout': 'Average_Monthly_Payout',
        'final_weighted_provider_score': 'Servicing_Provider_Anomaly_Score', 'score_provider': 'Provider_Perspective_Score',
        'score_patient': 'Patient_Perspective_Score', 'score_visit': 'Visit_Perspective_Score',
        'score_claim': 'Claim_Perspective_Score'})
    referrals['Category'] = normalize_category(referrals['Category'])
    referrals = referrals[~((referrals['Category'] == 'BH PROFESSIONAL')
                            & (pd.to_numeric(referrals['Average_Monthly_Payout'], errors='coerce') < 20000))]
    REFERRAL_SOURCE = 'raw referral files'
else:
    raise FileNotFoundError(REFERRAL_FILE)

for c in REFERRAL_COLS:
    if c not in referrals.columns:
        if c == 'Servicing_Provider_Anomaly_Score' or c in (provider_id, 'Category'):
            raise ValueError(f"Referral data has no {c}; columns: {list(referrals.columns)}")
        referrals[c] = np.nan
referrals = referrals[REFERRAL_COLS].copy()
referrals[provider_id] = normalize_provider_id(referrals[provider_id])
referrals['Category'] = normalize_category(referrals['Category'])
for c in REFERRAL_COLS[2:]:
    referrals[c] = pd.to_numeric(referrals[c], errors='coerce')
referrals = referrals.dropna(subset=[provider_id])
_dups = int(referrals.duplicated([provider_id, 'Category']).sum())
if _dups:
    print(f"NOTE: {_dups} repeated provider-category referral rows; keeping the highest score.")
referrals = (referrals.sort_values('Servicing_Provider_Anomaly_Score', ascending=False)
             .drop_duplicates([provider_id, 'Category']).reset_index(drop=True))

# Process referred providers only
_ref_keys = set(zip(referrals[provider_id], referrals['Category']))
_imp_keys = set(zip(imp[provider_id], imp['Category']))
load_stats['referred_provider_categories'] = len(_ref_keys)
load_stats['importance_only_not_referred'] = len(_imp_keys - _ref_keys)
load_stats['referred_without_feature_rows'] = len(_ref_keys - _imp_keys)
imp = imp[[k in _ref_keys for k in zip(imp[provider_id], imp['Category'])]].copy()
print(f"Referral source: {REFERRAL_SOURCE}")
print(f"Referred provider-categories: {len(_ref_keys)} | with feature rows: {len(_ref_keys & _imp_keys)} | "
      f"in importance file but not referred (skipped): {len(_imp_keys - _ref_keys)} | "
      f"referred with no feature rows: {len(_ref_keys - _imp_keys)}")
print(referrals['Category'].value_counts().to_string())

##### Load scoring data (peer comparisons)

In [ ]:
# ── Scoring files (all providers in each category): peer comparisons and provider values ──
# Only the columns needed are read: claim keys and amounts, plus any column named like a dictionary root.
BASE_SCORING_COLS = ['SProv_ID', provider_id, 'UNIQUE_KEY', 'claim_id', 'MEMBER_KEY', 'PMT_AMT', 'QUANTITY_PAID',
                     'PROC_CD', 'proc_cd_category', 'diag_1_level_I', 'ENC_CLM_IND']
_needed_roots = {match_root(f) for f in imp['Feature'].unique()} - {''}
_wanted = set(BASE_SCORING_COLS) | {r for r in _needed_roots if r not in COUNT_LEVEL_ROOTS}
_frames, scoring_stats = [], Counter()
for cat, (folder, _) in CATEGORY_FOLDERS.items():
    for stype in SCORING_FILE_TYPES:
        p = f'{PROJECTS}/{folder}/scoring_data/{stype}_monthly_scoring_file.csv.gz'
        if not os.path.exists(p):
            print(f"  missing scoring file: {p}")
            continue
        header = pd.read_csv(p, compression='gzip', nrows=0).columns
        cols = [c for c in header if c in _wanted]
        part = pd.read_csv(p, compression='gzip', usecols=cols, low_memory=False)
        part['Category'] = cat
        part['_file'] = stype
        _frames.append(part)
        scoring_stats[f'lines_{stype}'] += len(part)
df_scoring_master = pd.concat(_frames, ignore_index=True)
del _frames
if 'SProv_ID' in df_scoring_master.columns:
    if provider_id in df_scoring_master.columns:
        df_scoring_master[provider_id] = df_scoring_master[provider_id].fillna(df_scoring_master['SProv_ID'])
        df_scoring_master = df_scoring_master.drop(columns=['SProv_ID'])
    else:
        df_scoring_master = df_scoring_master.rename(columns={'SProv_ID': provider_id})
df_scoring_master[provider_id] = normalize_provider_id(df_scoring_master[provider_id])
df_scoring_master = df_scoring_master.dropna(subset=[provider_id])
# The 6-month and early files can hold the same claim line; keep it once (6-month file first)
if 'UNIQUE_KEY' in df_scoring_master.columns:
    _n = len(df_scoring_master)
    df_scoring_master = df_scoring_master.drop_duplicates(['Category', 'UNIQUE_KEY'], keep='first')
    scoring_stats['duplicate_lines_removed'] = _n - len(df_scoring_master)
else:
    print("WARNING: no UNIQUE_KEY column; claim lines present in both scoring files are counted twice.")
if 'claim_id' not in df_scoring_master.columns:
    df_scoring_master['claim_id'] = df_scoring_master.get('UNIQUE_KEY', pd.Series(range(len(df_scoring_master))))
df_scoring_master = df_scoring_master.drop(columns=['_file']).reset_index(drop=True)
_keys = {provider_id, 'UNIQUE_KEY', 'claim_id', 'MEMBER_KEY', 'Category'}
for c in df_scoring_master.columns:          # text columns as category: much less memory
    if c not in _keys and df_scoring_master[c].dtype == object:
        df_scoring_master[c] = df_scoring_master[c].astype('category')
print(f"Scoring lines: {len(df_scoring_master):,} | columns read: {df_scoring_master.shape[1]} | {dict(scoring_stats)}")
print(df_scoring_master.groupby('Category')[provider_id].nunique().rename('providers').to_string())

##### Peer comparison

In [ ]:
# ── Provider size and payment figures vs all providers in the same claim category (computed once) ──
PROFILE_METRICS = [   # (column, label, format)
    ('total_paid', 'Total paid', 'Dollars'),
    ('total_claims', 'Paid claims', 'Count'),
    ('unique_patients', 'Patients with paid claims', 'Count'),
    ('avg_paid_per_claim', 'Average paid per claim', 'Dollars'),
    ('claims_per_patient', 'Paid claims per patient', 'Ratio'),
    ('lines_per_claim', 'Paid lines per claim', 'Ratio'),
]
cohort_stats = {}
for cat, grp in df_scoring_master.groupby('Category'):
    g = grp.groupby(provider_id)
    st = pd.DataFrame({
        'total_paid': g['PMT_AMT'].sum() if 'PMT_AMT' in grp else np.nan,
        'total_claims': g['claim_id'].nunique(),
        'unique_patients': g['MEMBER_KEY'].nunique() if 'MEMBER_KEY' in grp else np.nan,
        'total_lines': g.size(),
    })
    st['avg_paid_per_claim'] = st['total_paid'] / st['total_claims'].replace(0, np.nan)
    st['claims_per_patient'] = st['total_claims'] / st['unique_patients'].replace(0, np.nan)
    st['lines_per_claim'] = st['total_lines'] / st['total_claims'].replace(0, np.nan)
    cohort_stats[cat] = st
print("Peer cohorts:", {c: len(s) for c, s in cohort_stats.items()})


def peer_comparison_lines(pid, cat):
    """Provider's own figures with the peer median and percentile; [] when the provider has no scoring lines."""
    st = cohort_stats.get(cat)
    if st is None or pid not in st.index or len(st) < MIN_PEERS:
        return []
    out = [f"Compared with {len(st)} {cat} providers (paid claims, same period):"]
    for col, label, vtype in PROFILE_METRICS:
        v = st.at[pid, col]
        if pd.isna(v):
            continue
        out.append(f"  {label}: {fmt_value(v, vtype)} for this provider; peer median "
                   f"{fmt_value(st[col].median(), vtype)}; {pct_ordinal(percentile_of(st[col], v))} percentile")
    return out


def main_services(pid, cat, n=3):
    """Top procedure categories by share of the provider's paid lines."""
    if 'proc_cd_category' not in df_scoring_master.columns:
        return ''
    lines = df_scoring_master.loc[(df_scoring_master['Category'] == cat) & (df_scoring_master[provider_id] == pid),
                                  'proc_cd_category'].dropna()
    if lines.empty:
        return ''
    vc = lines.value_counts(normalize=True).head(n)
    return ', '.join(f"{k} ({v:.0%} of paid lines)" for k, v in vc.items())

##### Provider values (Python-computed)

In [ ]:
# ── Provider values for Value_Template sentences (Python only; the LLM copies them) ──
# Filled only when the scoring files hold the measure unambiguously:
#   A. one-hot level of a claim-line column (code, category, tier, yes/no): share of the provider's paid lines
#   B. a numeric column constant within the provider and within almost every provider (a provider-level measure)
# Peers = all providers in the same claim category with the value. Standardized scores are never quoted.
NON_NEGATIVE_TYPES = {'Count', 'Days', 'Miles', 'Units', 'Dollars'}
UNIT_INTERVAL_TYPES = {'Share (0-1)', 'Index (0-1)'}
NEVER_QUOTE_TYPES = {'Standardized score', 'Date part', 'Count level'}
_YES = {'1', 'y', 'yes', 'true', 't'}
_NO = {'0', 'n', 'no', 'false', 'f'}


def _norm_one(x):
    return re.sub(r'^(-?\d+)\.0+$', r'\1', str(x).strip().lower()).replace(' ', '_')


def _norm_level_series(col):
    """Normalized level labels; for category columns only the (few) categories are normalized."""
    if isinstance(col.dtype, pd.CategoricalDtype):
        return col.map({c: _norm_one(c) for c in col.cat.categories})
    return col.astype(str).map(_norm_one)


def _flag_series(col):
    s = col.astype(str).str.strip().str.lower().str.replace(r'^(-?\d+)\.0+$', r'\1', regex=True)
    return s.map(lambda x: 1.0 if x in _YES else (0.0 if x in _NO else np.nan))


def _looks_scaled(vtype, per_provider):
    v = per_provider.dropna()
    if v.empty:
        return True
    if vtype in UNIT_INTERVAL_TYPES:
        return bool(((v < 0) | (v > 1)).any())
    if vtype in NON_NEGATIVE_TYPES:
        return bool((v < 0).any() or (v.between(0, 1).all() and v.nunique() > 2))
    return False


def build_provider_feature_values(needed):
    """needed: DataFrame [provider_id, 'Category', 'Feature', 'Feature_Root'].
    Returns (values_df [provider_id, Category, Feature, value, peer_median, peer_pct, n_claims, value_source], coverage)."""
    rows, coverage = [], Counter()
    for cat, need_cat in needed.groupby('Category'):
        grp = df_scoring_master[df_scoring_master['Category'] == cat]
        if grp.empty:
            coverage['category_not_in_scoring_data'] += need_cat['Feature'].nunique()
            continue
        pids = grp[provider_id]
        n_claims = grp.groupby(provider_id)['claim_id'].nunique()
        norm_cache = {}
        for (feat, root), need_f in need_cat.groupby(['Feature', 'Feature_Root']):
            meta = MAP[root]
            vtype, measured_at = meta['Value_Type'], meta['Measured_At']
            if vtype in NEVER_QUOTE_TYPES or not meta['Value_Template']:
                coverage['not_quoted_by_design'] += 1
                continue
            if root not in grp.columns:
                coverage['column_not_in_scoring_data'] += 1
                continue
            col = grp[root]
            if vtype in ONE_HOT_VALUE_TYPES:
                if measured_at != 'Claim line':
                    coverage['share_of_lines_would_mislabel'] += 1
                    continue
                if vtype == 'Yes/No flag':      # the template describes the 'yes' condition, also for a '_0' feature
                    ind = _flag_series(col) == 1
                else:
                    if root not in norm_cache:
                        norm_cache[root] = _norm_level_series(col)
                    ind = norm_cache[root] == _norm_one(raw_level(feat, root))
                if not ind.any():
                    coverage['level_not_found_in_column'] += 1
                    continue
                per_provider = ind.groupby(pids).mean()
                source = 'share_of_lines'
            else:
                num = pd.to_numeric(col, errors='coerce')
                if num.notna().sum() == 0:
                    num = _flag_series(col)
                spread = num.groupby(pids).nunique(dropna=True)
                if (spread > 1).mean() > 0.01:
                    coverage['varies_within_provider'] += 1
                    continue
                per_provider = num.groupby(pids).first().dropna()
                if _looks_scaled(vtype, per_provider):
                    coverage['out_of_range_looks_scaled'] += 1
                    continue
                source = 'provider_measure'
            if len(per_provider) < MIN_PEERS:
                coverage['too_few_peers'] += 1
                continue
            peer_median = float(per_provider.median())
            for pid in need_f[provider_id].unique():
                if pid not in per_provider.index:
                    coverage['provider_not_in_scoring_data'] += 1
                    continue
                if source == 'provider_measure' and spread.get(pid, 0) > 1:
                    coverage['varies_within_this_provider'] += 1      # this provider's own lines disagree
                    continue
                v = float(per_provider.at[pid])
                rows.append({provider_id: pid, 'Category': cat, 'Feature': feat, 'value': v,
                             'peer_median': peer_median, 'peer_pct': percentile_of(per_provider, v),
                             'n_claims': int(n_claims.get(pid, 0)), 'value_source': source})
                coverage[source] += 1
    cols = [provider_id, 'Category', 'Feature', 'value', 'peer_median', 'peer_pct', 'n_claims', 'value_source']
    return pd.DataFrame(rows, columns=cols), coverage


def render_value_sentence(row):
    """Fill the dictionary Value_Template; '' unless every placeholder can be filled honestly."""
    tpl = MAP[row['Feature_Root']]['Value_Template']
    if not tpl or pd.isna(row.get('value')):
        return ''
    if row.get('n_claims', 0) < MIN_CLAIMS_FOR_VALUE:
        return ''
    vtype = MAP[row['Feature_Root']]['Value_Type']
    vals = {'value': fmt_value(row['value'], vtype), 'peer_median': fmt_value(row['peer_median'], vtype),
            'peer_pct': pct_ordinal(row['peer_pct']) if not pd.isna(row['peer_pct']) else None,
            'level': row.get('Level_Plain') or None, 'category': CATEGORY_DISPLAY.get(row['Category'], row['Category'])}
    needed = set(re.findall(r'\{(\w+)\}', tpl))
    if any(vals.get(k) in (None, '') for k in needed):
        return ''
    out = tpl.format(**{k: vals[k] for k in needed})
    if vtype == 'Units' and 'wait' in row['Feature'].lower() and 'unit' in row['Feature'].lower():
        hrs = float(row['value']) * 0.5                 # T2007: 1 unit = 30 minutes
        out = out.replace(f"{vals['value']} ", f"{vals['value']} (about {hrs:,.1f} hours) ", 1)
    return out

##### Prepare features
Coverage gate, roles, values, main / supporting / background patterns, patient mix, one row per referred provider.

In [ ]:
# ── Prepare features for every referred provider ──
prep_stats = Counter()
_k = [provider_id, 'Category', 'Feature']

# (1) One row per feature: the strongest analysis level is kept, with the levels it appeared at
imp = imp.sort_values(_k + ['Feature_Rank_The_Provider', 'Overall_Importance_Score'],
                      ascending=[True, True, True, True, False], kind='stable')
_STAB_RANK = {'persistent': 4, 'emerging': 3, 'sporadic': 2, 'declining': 1, 'unknown': 0}
_CONS_RANK = {'unanimous': 4, 'strong': 3, 'partial': 2, 'weak': 1, 'unknown': 0}
_lv = imp.groupby(_k).agg(Levels_In_File=('Analysis_Level', lambda s: ','.join(sorted(set(map(str, s))))),
                          In_Main_Tier=('Importance_Tier', lambda s: bool(set(s) & MAIN_TIERS)),
                          Stability=('Temporal_Stability', lambda s: max(s, key=lambda x: _STAB_RANK.get(x, 0))),
                          Consensus=('Model_Consensus', lambda s: max(s, key=lambda x: _CONS_RANK.get(x, 0))),
                          N_Levels=('Analysis_Level', 'nunique')).reset_index()
feat = imp.drop_duplicates(_k, keep='first').merge(_lv, on=_k, how='left', validate='1:1')
feat = feat.rename(columns={'Analysis_Level': 'Best_Level'})
prep_stats['feature_rows_all_levels'] = len(imp)
prep_stats['feature_rows'] = len(feat)

# (2) Coverage gate: every feature needs a dictionary row
_roots = {f: match_root(f) for f in feat['Feature'].unique()}
feat['Feature_Root'] = feat['Feature'].map(_roots)
_unmapped = sorted(f for f, r in _roots.items() if not r)
if _unmapped:
    _path = f'{OUTPUT_DIR}/llm_summary_unmapped_features_{scoring_date}.csv'
    (feat[feat['Feature_Root'] == ''][['Feature', 'Feature_Category']].drop_duplicates()
        .to_csv(_path, index=False))
    msg = (f"{len(_unmapped)} features have no dictionary row (list saved to {_path}); add them to the dictionary:\n  "
           + "\n  ".join(_unmapped[:30]))
    if STRICT_COVERAGE:
        raise ValueError(msg)
    print("WARNING:", msg)
    prep_stats['dropped_unmapped'] = int((feat['Feature_Root'] == '').sum())
    feat = feat[feat['Feature_Root'] != '']

# (3) Dictionary fields, plain labels, decoded codes
for c in ['Narrate_Role', 'Validation_Status', 'Applies_To', 'Concept_Group', 'Value_Type', 'Measured_At',
          'Scheme_Link', 'Concern_Direction', 'Description', 'Suspicious_Pattern', 'Benign_Explanations',
          'Narrative_Caveat', 'Records_To_Review', 'Policy_Lever', 'Definition_Confidence', 'Level_Dependent']:
    feat[c] = feat['Feature_Root'].map(lambda r, c=c: MAP[r][c])
_levels = {(f, r): plain_level(f, r) for f, r in feat[['Feature', 'Feature_Root']].drop_duplicates().itertuples(index=False)}
feat['Level_Plain'] = [_levels[(f, r)][0] for f, r in zip(feat['Feature'], feat['Feature_Root'])]
feat['Level_Missing'] = [_levels[(f, r)][1] for f, r in zip(feat['Feature'], feat['Feature_Root'])]
feat['Label'] = [feature_label(r, lv) for r, lv in zip(feat['Feature_Root'], feat['Level_Plain'])]
for c in ['Description']:      # count-level rows carry {level} in their text
    feat[c] = [t.replace('{level}', lv) for t, lv in zip(feat[c], feat['Level_Plain'])]
_codes = {f: decode_feature_codes(f) for f in feat['Feature'].unique()}
feat['Code_Decoded'] = feat['Feature'].map(_codes)
feat['Is_Demographic'] = feat['Feature_Category'].isin(DEMOGRAPHIC_CATEGORIES)

# (4) Effective role: what the note may use this feature for
feat['Role'], feat['Role_Reason'] = feat['Narrate_Role'], ''
def _set_role(mask, role, reason):
    m = mask & (feat['Role_Reason'] == '')
    feat.loc[m, 'Role'] = role
    feat.loc[m, 'Role_Reason'] = reason
    prep_stats[f'role:{reason}'] += int(m.sum())
_set_role(feat['Narrate_Role'] == 'Do not narrate', 'Not narrated', 'do not narrate')
_set_role(feat['Validation_Status'] == 'Hold until confirmed', 'Not narrated', 'on hold')
_set_role(pd.Series([not applies_to(a, c) for a, c in zip(feat['Applies_To'], feat['Category'])], index=feat.index),
          'Not narrated', 'not applicable to category')
_set_role(feat['Level_Missing'], 'Not narrated', 'missing-value level')
_set_role((feat['Role'] == 'Driver') & feat['Level_Plain'].isin(NON_CONCERNING_LEVELS), 'Context only', 'level not concerning')

# (5) Provider values (Python only), for features the note may use
_needed = feat.loc[feat['Role'].isin(['Driver', 'Context only']), [provider_id, 'Category', 'Feature', 'Feature_Root']]
provider_feature_values, value_coverage = build_provider_feature_values(_needed.drop_duplicates())
feat = feat.merge(provider_feature_values, on=_k, how='left', validate='1:1')
_few = feat['n_claims'].notna() & (feat['n_claims'] < MIN_CLAIMS_FOR_VALUE)
feat.loc[_few, ['value', 'peer_median', 'peer_pct']] = np.nan
prep_stats['values_dropped_few_claims'] = int(_few.sum())
# A level-dependent measure whose strongest signal came from another analysis level: the provider-level value from the
# scoring files would describe a different calculation, so it is not quoted (and cannot confirm a scheme)
_other_level = (feat['Level_Dependent'] == 'Yes') & (feat['Best_Level'].astype(str).str.lower() != 'provider') \
    & (feat['value_source'] == 'provider_measure')
feat.loc[_other_level, ['value', 'peer_median', 'peer_pct']] = np.nan
prep_stats['values_dropped_other_analysis_level'] = int(_other_level.sum())
feat['Value_Sentence'] = [render_value_sentence(r) if role in ('Driver', 'Context only') else ''
                          for r, role in zip(feat.to_dict('records'), feat['Role'])]


def scrub_brief_text(text):
    """Replace words the notes may not use (BRIEF_WORD_SCRUB) in dictionary text before it reaches the LLM."""
    t = str(text or '')
    for pat, rep in BRIEF_WORD_SCRUB:
        t = re.sub(pat, rep, t, flags=re.I)
    return t


brief_scrub_hits = Counter()
for _c in ['Description', 'Suspicious_Pattern', 'Benign_Explanations', 'Narrative_Caveat', 'Value_Sentence']:
    for pat, _ in BRIEF_WORD_SCRUB:
        _n = int(feat[_c].astype(str).map(lambda t: bool(re.search(pat, t, re.I))).sum())
        if _n:
            brief_scrub_hits[f'{_c}:{pat}'] += _n
    feat[_c] = feat[_c].map(scrub_brief_text)
prep_stats['brief_words_scrubbed'] = int(sum(brief_scrub_hits.values()))

# (6) A driver whose measured value sits on the harmless side of peers is background, never a reason
_harmless = (feat['Role'] == 'Driver') & feat['peer_pct'].notna() & (
    ((feat['Concern_Direction'] == 'High') & (feat['peer_pct'] < HARMLESS_PCT_HIGH))
    | ((feat['Concern_Direction'] == 'Low') & (feat['peer_pct'] > HARMLESS_PCT_LOW)))
feat.loc[_harmless, 'Role'] = 'Context only'
feat.loc[_harmless, 'Role_Reason'] = 'harmless side of peers'
prep_stats['role:harmless side of peers'] = int(_harmless.sum())

# (7) What the LLM sees: MAIN (drivers inside the 80% signal coverage), SUPPORTING (next drivers), BACKGROUND.
#     One line per Concept_Group; different codes of the same field stay separate (they are different evidence).
feat = feat.sort_values([provider_id, 'Category', 'Feature_Rank_The_Provider'], kind='stable').reset_index(drop=True)
feat['_key'] = np.where(feat['Concept_Group'] != '', feat['Concept_Group'], feat['Feature_Root'])
_code_level = (feat['Value_Type'] == 'Category level') & (feat['Feature'] != feat['Feature_Root'])
feat.loc[_code_level, '_key'] = feat.loc[_code_level, '_key'] + '|' + feat.loc[_code_level, 'Label']
feat['Shown_As'] = ''
for _, g in feat.groupby([provider_id, 'Category'], sort=False):
    used, n_main, n_sup, n_bg = set(), 0, 0, 0
    for i, r in g[g['Role'] == 'Driver'].iterrows():
        if (r['In_Main_Tier'] and n_main < MAX_MAIN and r['_key'] not in used
                and CONFIDENCE_ORDER.index(r['Definition_Confidence']) >= CONFIDENCE_ORDER.index(MIN_CONFIDENCE_FOR_MAIN)):
            feat.at[i, 'Shown_As'] = 'Main'; used.add(r['_key']); n_main += 1
    for i, r in g[(g['Role'] == 'Driver') & (feat.loc[g.index, 'Shown_As'] == '')].iterrows():
        if n_sup < MAX_SUPPORTING and r['_key'] not in used:
            feat.at[i, 'Shown_As'] = 'Supporting'; used.add(r['_key']); n_sup += 1
    for i, r in g[g['Role'] == 'Context only'].iterrows():
        if n_bg < MAX_BACKGROUND and r['_key'] not in used:
            feat.at[i, 'Shown_As'] = 'Background'; used.add(r['_key']); n_bg += 1
# Evidence items for the Python note of non-narrated providers: main, then supporting, then background, in rank order
feat['Evidence_Rank'] = np.nan
_ev_order = {'Main': 0, 'Supporting': 1, 'Background': 2}
for _, g in feat[feat['Shown_As'].isin(_ev_order)].groupby([provider_id, 'Category'], sort=False):
    g2 = g.assign(_o=g['Shown_As'].map(_ev_order)).sort_values(['_o', 'Feature_Rank_The_Provider'], kind='stable')
    for n, i in enumerate(g2.index[:MAX_EVIDENCE_ITEMS], start=1):
        feat.at[i, 'Evidence_Rank'] = n

# (8) Patient mix: demographic share of the top-ranked signal, and whether a billing driver is among the strongest
_pk = [provider_id, 'Category']
_top = feat.groupby(_pk, sort=False).head(TOP_K_SIGNAL).copy()
_top['_demo_score'] = _top['Overall_Importance_Score'].where(_top['Is_Demographic'], 0.0)
_mix = _top.groupby(_pk)[['_demo_score', 'Overall_Importance_Score']].sum()
_mix['Patient_Mix_Share'] = (_mix['_demo_score'] / _mix['Overall_Importance_Score'].clip(lower=1e-12)).round(3)
_mix['Driver_In_Top'] = (feat.groupby(_pk, sort=False).head(PATIENT_MIX_TOP_DRIVERS)
                         .assign(_d=lambda d: d['Role'] == 'Driver').groupby(_pk)['_d'].any())
_mix['Mix_Labels'] = (_top[_top['Is_Demographic']].assign(_lab=lambda d: d['Feature_Root'].map(lambda r: MAP[r]['Replacement']))
                      .groupby(_pk)['_lab'].agg(lambda s: '; '.join(dict.fromkeys(s))))
_mix = _mix.drop(columns=['_demo_score', 'Overall_Importance_Score']).reset_index()
_mix['Mix_Labels'] = _mix['Mix_Labels'].fillna('')
_mix['Driver_In_Top'] = _mix['Driver_In_Top'].fillna(False).astype(bool)
_mix['Patient_Mix'] = (_mix['Patient_Mix_Share'] >= PATIENT_MIX_SHARE) & ~_mix['Driver_In_Top']

# (9) One row per referred provider
_counts = (feat.groupby([provider_id, 'Category'])['Shown_As'].value_counts().unstack(fill_value=0)
           .reindex(columns=['Main', 'Supporting', 'Background'], fill_value=0)
           .rename(columns=lambda c: f'n_{c.lower()}').reset_index())
providers = (referrals.merge(_counts, on=[provider_id, 'Category'], how='left')
             .merge(_mix, on=[provider_id, 'Category'], how='left'))
_has_rows = set(zip(feat[provider_id], feat['Category']))
providers['Has_Features'] = [k in _has_rows for k in zip(providers[provider_id], providers['Category'])]
for c in ['n_main', 'n_supporting', 'n_background']:
    providers[c] = providers[c].fillna(0).astype(int)
providers['Patient_Mix'] = providers['Patient_Mix'].fillna(False).astype(bool)
providers['Note_Type'] = np.select(
    [~providers['Has_Features'], providers['Patient_Mix'], providers['n_main'] == 0],
    ['evidence_only', 'patient_mix', 'evidence_only'], default='narrative_eligible')
providers['Note_Reason'] = np.select(
    [~providers['Has_Features'], providers['Patient_Mix'], providers['n_main'] == 0],
    ['no feature rows', 'patient mix', 'no billing measure in the 80% signal coverage'], default='')

features_by_provider = {k: g for k, g in feat.groupby([provider_id, 'Category'], sort=False)}
print("Feature preparation:", dict(prep_stats))
print("Value sentences:", dict(value_coverage))
print(providers.groupby('Category')['Note_Type'].value_counts().unstack(fill_value=0).to_string())

##### Narrative selection and deterministic schemes

In [ ]:
# ── Schemes from the dictionary (Scheme_Link of the MAIN patterns), gated by the provider's measured value ──
def _links(r):
    return [s.strip().lower() for s in str(r['Scheme_Link']).split('|') if s.strip() and s.strip().lower() not in ('none', 'other')]


def derive_schemes(main_rows):
    """Deterministic scheme pre-classification for one provider.
    confirmed: {scheme: [pattern rows]} where the measured value is far on the concerning side of peers
               (High: at or above SCHEME_PCT_HIGH; Low: at or below SCHEME_PCT_LOW)
    possible:  linked schemes whose value is only moderately on the concerning side (beyond SCHEME_PCT_NEAR),
               without a measured value, or an 'Either' measure at either extreme
    strength (v9 scale, computed from the confirmed patterns):
      high   = 2+ patterns, at least one persistent, from 2+ analysis levels
      medium = 1 persistent pattern, or 2+ patterns
      low    = 1 pattern, not persistent
    Returns (confirmed_list, possible_list, rationale_text) with confirmed_list = [{'scheme','evidence','rationale','labels'}]."""
    confirmed, possible, why = {}, {}, []
    for _, r in main_rows.iterrows():
        links = _links(r)
        if not links:
            continue
        pct, d = r.get('peer_pct'), r['Concern_Direction']
        measured = pct is not None and not pd.isna(pct)
        if measured and ((d == 'High' and pct >= SCHEME_PCT_HIGH) or (d == 'Low' and pct <= SCHEME_PCT_LOW)):
            target, how = confirmed, f"{pct_ordinal(pct)} percentile"
        elif measured and ((d == 'High' and pct >= SCHEME_PCT_NEAR) or (d == 'Low' and pct <= 100 - SCHEME_PCT_NEAR)):
            target, how = possible, f"{pct_ordinal(pct)} percentile, not far enough from peers to confirm"
        elif measured and d == 'Either' and (pct >= SCHEME_PCT_HIGH or pct <= SCHEME_PCT_LOW):
            target, how = possible, f"{pct_ordinal(pct)} percentile, direction not defined"
        elif not measured:
            target, how = possible, 'no provider value'
        else:
            continue
        for lab in links:
            target.setdefault(lab, []).append(r)
        why.append(f"{r['Label']} ({how})")
    # one measure alone cannot confirm these labels
    for lab in [l for l in confirmed if l in SCHEMES_NEEDING_TWO_MEASURES and len({x['Feature_Root'] for x in confirmed[l]}) < 2]:
        possible.setdefault(lab, []).extend(confirmed.pop(lab))
    for lab in confirmed:
        possible.pop(lab, None)
    out = []
    for lab, rows in confirmed.items():
        roots = {x['Feature_Root'] for x in rows}
        persistent = any(x.get('Stability') == 'persistent' for x in rows)
        levels = {str(x.get('Best_Level', '')).lower() for x in rows}
        if len(roots) >= 2 and persistent and len(levels) >= 2:
            strength = 'high'
        elif persistent or len(roots) >= 2:
            strength = 'medium'
        else:
            strength = 'low'
        labels = list(dict.fromkeys(x['Label'] for x in rows))
        out.append({'scheme': lab, 'evidence': strength, 'labels': labels,
                    'rationale': f"{'; '.join(labels)} at or beyond the {pct_ordinal(SCHEME_PCT_HIGH)} percentile of peers (or the {pct_ordinal(SCHEME_PCT_LOW)} for low-side measures)"
                                 + ('; persistent across scoring windows' if persistent else '')})
    out.sort(key=lambda x: ({'high': 0, 'medium': 1, 'low': 2}[x['evidence']], x['scheme']))
    return out, list(possible), '; '.join(why)


def scheme_block(key, note_type):
    """Deterministic scheme result for a provider: dict with detected, possible, rationale, candidates, allowed."""
    if note_type == 'patient_mix':
        return dict(detected='Not assessed (patient mix)', possible='', rationale='', candidates=[], allowed=[])
    g = features_by_provider.get(key)
    main = g[g['Shown_As'] == 'Main'] if g is not None else None
    if main is None or main.empty:
        return dict(detected='Not assessed (no billing measure to narrate)', possible='', rationale='', candidates=[], allowed=[])
    cands, possible, why = derive_schemes(main)
    any_link = any(_links(r) for _, r in main.iterrows())
    if cands:
        detected = ', '.join(f"{c['scheme']} ({c['evidence']} evidence)" for c in cands)
    elif possible:
        detected = 'Not confirmed (see Possible_Schemes)'
    elif not any_link:
        detected = NEW_PATTERN_LABEL
    else:
        detected = 'No scheme confirmed (linked measures not far enough from peers)'
    return dict(detected=detected, possible=', '.join(possible), rationale=why, candidates=cands,
                allowed=[c['scheme'] for c in cands if c['scheme'] in SCHEME_DEFINITIONS_BY_LABEL])


providers['_key'] = list(zip(providers[provider_id], providers['Category']))
_sb = {k: scheme_block(k, t) for k, t in zip(providers['_key'], providers['Note_Type'])}
providers['Scheme_Strength'] = [max((c['evidence'] for c in _sb[k]['candidates']), key=lambda e: {'high': 2, 'medium': 1, 'low': 0}[e], default='')
                                for k in providers['_key']]

# ── Which referred providers get an LLM narrative: v9 quota (share of each category, with a floor) on eligible providers ──
# Patient-mix and no-main-pattern providers never take a slot. Ties broken by provider ID so reruns select the same set.
providers = providers.sort_values(['Category', 'Servicing_Provider_Anomaly_Score', provider_id],
                                  ascending=[True, False, True], na_position='last', kind='stable').reset_index(drop=True)
providers['Category_Rank'] = providers.groupby('Category').cumcount() + 1
_elig = providers['Note_Type'] == 'narrative_eligible'
providers['Eligible_Rank'] = np.nan
providers.loc[_elig, 'Eligible_Rank'] = providers[_elig].groupby('Category').cumcount() + 1
_n_elig = providers[_elig].groupby('Category')[provider_id].transform('size')
_quota = np.minimum(_n_elig, np.maximum(NARRATE_FLOOR, np.ceil(NARRATE_SHARE * _n_elig)))
providers['Quota'] = np.nan
providers.loc[_elig, 'Quota'] = _quota
providers['Narrate_Selected'] = _elig & (providers['Eligible_Rank'] <= providers['Quota'])
if NARRATE_HIGH_EVIDENCE:
    providers['Narrate_Selected'] |= _elig & (providers['Scheme_Strength'] == 'high')
if NARRATE_CAP:
    providers['Narrate_Selected'] &= providers['Eligible_Rank'] <= NARRATE_CAP
providers.loc[_elig & ~providers['Narrate_Selected'], 'Note_Type'] = 'evidence_list'
providers.loc[providers['Note_Type'] == 'evidence_list', 'Note_Reason'] = 'below the narrative quota for its category'
providers.loc[providers['Narrate_Selected'], 'Note_Type'] = 'narrative'

print(f"Selection: top {NARRATE_SHARE:.0%} of eligible providers per category, floor {NARRATE_FLOOR}, cap {NARRATE_CAP}, "
      f"high-evidence override {NARRATE_HIGH_EVIDENCE}")
print(providers.groupby('Category').agg(referred=('Note_Type', 'size'), eligible=('Eligible_Rank', 'count'),
                                        narrative=('Narrate_Selected', 'sum')).to_string())
print("Note types:", providers['Note_Type'].value_counts().to_dict())
print("Deterministic scheme strength:", providers['Scheme_Strength'].replace('', 'none').value_counts().to_dict())

##### Evidence brief, prompts (narrative and Layer 2) and Python notes

In [ ]:
# ── Evidence brief (Python): the story elements every note is built from ──
_SCHEME_CLAUSE = re.compile(r'[,;]?\s*consistent with [^.;]*|\s*\(possible [^)]*\)', re.I)
_SCHEME_NAMES = sorted([s for s in KNOWN_SCHEMES if s != 'other'] + ['upcod', 'unbundl', 'phantom', 'duplicate claim'],
                       key=len, reverse=True)


def _sentence(s):
    s = str(s or '').strip()
    return s if not s or s.endswith(('.', '!', '?')) else s + '.'


def _without_scheme_sentences(text, allowed):
    """Drop dictionary sentences that name a scheme (or a synonym of one) the note may not use."""
    keep = []
    for s in re.split(r'(?<=[.;])\s+', str(text or '')):
        low = s.lower()
        if any(n in low and not any(n in a for a in allowed) for n in _SCHEME_NAMES):
            continue
        if any(re.search(pat, low) and lab not in allowed for pat, lab in SCHEME_SYNONYMS):
            continue
        keep.append(s)
    return ' '.join(keep).strip()


def _solidity(r):
    """Stability and agreement of one pattern, in plain words (never 'model' words)."""
    parts = []
    stab = STABILITY_PHRASE.get(str(r.get('Stability', 'unknown')), '')
    if stab:
        parts.append(stab)
    n = r.get('N_Levels')
    if n is not None and not pd.isna(n) and n >= 2:
        parts.append(f"seen at {int(n)} analysis levels")
    return '; '.join(parts)


def _first_clause(text):
    """First item of a dictionary list field ('A, B, C' or 'A; B'), for a short closing sentence."""
    t = str(text or '').strip()
    if not t or t.lower() == 'none':
        return ''
    return re.split(r'[;.]', t)[0].strip().rstrip(',')


NO_VALUE_SENTENCE = ("No provider value is available for this measure in the claims extract: say only that it was unusual "
                     "compared with peers, with no number.")


def build_brief(pid, cat, g, schemes):
    """Everything a note may say about one provider, computed in Python."""
    main = g[g['Shown_As'] == 'Main']
    sup = g[g['Shown_As'] == 'Supporting']
    bg = g[g['Shown_As'] == 'Background']
    allowed = schemes['allowed']

    def pattern(r, full):
        d = {'label': r['Label'], 'description': _sentence(r['Description']),
             'value_sentence': _sentence(r['Value_Sentence']) if r['Value_Sentence'] else NO_VALUE_SENTENCE,
             'codes': r['Code_Decoded'], 'solidity': _solidity(r), 'level_phrase': '',
             'suspicious': '', 'benign': '', 'caveat': '', 'records': ''}
        if r.get('Level_Dependent') == 'Yes' and str(r.get('Best_Level', '')).lower() in LEVEL_PHRASE:
            d['level_phrase'] = LEVEL_PHRASE[str(r['Best_Level']).lower()]
        if full:
            d['suspicious'] = _sentence(_without_scheme_sentences(_SCHEME_CLAUSE.sub('', r['Suspicious_Pattern']).strip(), allowed))
            d['benign'] = _without_scheme_sentences(r['Benign_Explanations'], allowed)
            d['caveat'] = _without_scheme_sentences(r['Narrative_Caveat'], allowed)
            d['records'] = _first_clause(r['Records_To_Review'])
        return d

    brief = {
        'pid': pid, 'cat': cat, 'population': POPULATION_CONTEXT.get(cat, ''),
        'main': [pattern(r, True) for _, r in main.iterrows()],
        'supporting': [pattern(r, False) for _, r in sup.iterrows()],
        'background': [{'label': r['Label'], 'value_sentence': _sentence(r['Value_Sentence'])} for _, r in bg.iterrows()],
        'services': main_services(pid, cat),
        'peers': peer_comparison_lines(pid, cat),
        'schemes': schemes,
        'records': list(dict.fromkeys(p['records'] for p in [pattern(r, True) for _, r in main.iterrows()] if p['records']))[:2],
        'stability_summary': '',
    }
    brief['stability_summary'] = stability_summary([str(s) for s in main['Stability']] if len(main) else [])
    return brief


def stability_summary(stabs):
    """One sentence on how steady the main patterns are, by scoring window; mixed cases name the patterns by number."""
    known = [s for s in stabs if STABILITY_PHRASE.get(s)]
    if not known:
        return ''
    if len(set(known)) == 1 and len(known) == len(stabs):
        s = known[0]
        if len(stabs) == 1:
            return f"The main pattern was {STABILITY_PHRASE[s]}."
        return f"All main patterns were {STABILITY_PHRASE[s]}."
    groups = {}
    for k, s in enumerate(stabs, 1):
        if STABILITY_PHRASE.get(s):
            groups.setdefault(s, []).append(str(k))
    parts = []
    for s in ('persistent', 'emerging', 'declining', 'sporadic'):
        if s in groups:
            ks = groups[s]
            label = f"main pattern {ks[0]}" if len(ks) == 1 else 'main patterns ' + ', '.join(ks[:-1]) + ' and ' + ks[-1]
            parts.append(f"{label} {'was' if len(ks) == 1 else 'were'} {STABILITY_PHRASE[s]}")
    return (parts[0][0].upper() + parts[0][1:] + ('; ' + '; '.join(parts[1:]) if len(parts) > 1 else '') + '.')


def brief_text(brief, for_layer2=False):
    """The evidence brief as prompt text. for_layer2 adds nothing the narrative does not see (one source of facts)."""
    b = brief
    lines = [f"PROVIDER: {b['pid']} | CLAIM CATEGORY: {b['cat']}"]
    if b['population']:
        lines.append(f"POPULATION: {b['population']}")
    if b['services']:
        lines.append(f"MAIN SERVICES: {b['services']}")
    if b['peers']:
        lines += ['', 'PROVIDER SIZE AND PEER COMPARISON (context: not a reason for the flag; quote only next to a main pattern):'] + b['peers']
    lines += ['', 'MAIN PATTERNS (the reasons the provider stood out, strongest first):']
    for k, p in enumerate(b['main'], 1):
        head = f"  {k}. {p['label']}" + (f" ({p['level_phrase']})" if p['level_phrase'] else '') + (f" [{p['codes']}]" if p['codes'] else '')
        body = ' '.join(x for x in [p['description'], p['value_sentence'],
                                    _sentence(p['solidity'].capitalize()) if p['solidity'] else '',
                                    p['suspicious'],
                                    _sentence(f"Other explanations: {p['benign']}") if p['benign'] else '',
                                    _sentence(f"Note: {p['caveat']}") if p['caveat'] else ''] if x)
        lines.append(f"{head}: {body}")
    lines += ['', 'SUPPORTING PATTERNS (may add detail to the main patterns):']
    lines += [f"  {k}. {p['label']}" + (f" ({p['level_phrase']})" if p['level_phrase'] else '') + (f" [{p['codes']}]" if p['codes'] else '')
              + f": {' '.join(x for x in [p['description'], p['value_sentence'], _sentence(p['solidity'].capitalize()) if p['solidity'] else ''] if x)}"
              for k, p in enumerate(b['supporting'], 1)] or ['  None.']
    lines += ['', 'BACKGROUND (never a reason for the flag):']
    lines += [f"  {k}. {p['label']}" + (f": {p['value_sentence']}" if p['value_sentence'] else '')
              for k, p in enumerate(b['background'], 1)] or ['  None.']
    if b['stability_summary']:
        lines += ['', f"STABILITY: {b['stability_summary']}"]
    if b['records']:
        lines += ['', 'RECORDS THAT WOULD CLARIFY THE MAIN PATTERNS: ' + '; '.join(b['records']) + '.']
    return '\n'.join(lines)


# ── Narrative prompt (v9 skeleton: definitions, data boundary, ranked rules; v4 facts: Python-written numbers only) ──
NO_CLINICAL_DATA = ("DATA BOUNDARY: you have paid-claims data only. No medical records, prior authorizations or clinical "
                    "documentation. Never say a service was or was not medically necessary, appropriate or justified, and never "
                    "say a service was or was not delivered. Paid is the only fact.")


def build_prompt(brief, allowed_schemes, reasoning_level):
    b = brief
    if allowed_schemes:
        scheme_rule = ("- A main pattern may be described as 'consistent with' one of these schemes, and only these: "
                       + '; '.join(allowed_schemes) + ". Never say a scheme happened or was intended.\n")
        scheme_defs = "SCHEME DEFINITIONS (paid-claims wording):\n" + '\n'.join(f"- {s}: {SCHEME_DEFINITIONS_BY_LABEL[s]}"
                                                                                for s in allowed_schemes) + '\n\n'
    else:
        scheme_rule = ("- Do not name any billing scheme. After the main patterns, write exactly: "
                       "'The main patterns match no known billing scheme.'\n")
        scheme_defs = ''
    system_msg = (
        "You write short case notes for Medicaid program-integrity investigators. The note is the starting point of an "
        "investigation: it must say what stood out, by how much, how steady it is, what it could mean, what else could "
        "explain it, and which records would settle it.\n\n"
        + NO_CLINICAL_DATA + "\n\n"
        "MUST (never violate):\n"
        "- Use only facts and numbers in the user message. Do not calculate new numbers, ratios, multiples or differences, "
        "and do not round a number into a different one. Every number must stay next to the measure it belongs to.\n"
        "- All figures come from PAID claims: write 'paid'.\n"
        "- Say the provider is higher or lower than peers only where the message gives a provider value with a peer median, "
        "or a percentile. Otherwise say the pattern was unusual compared with peers.\n"
        "- Describe the MAIN PATTERNS in order (at most 3), each with its own value. SUPPORTING items may add detail to them. "
        "BACKGROUND items may be mentioned once, introduced with 'In addition,', and never as a reason for the flag.\n"
        "- The provider's size (total paid, number of claims or patients) is context, never the reason for the flag; "
        "quote a size figure only next to the main pattern it puts in proportion.\n"
        + scheme_rule +
        "- Code descriptions only identify codes. Do not judge medical necessity, quality of care, or whether a diagnosis "
        "supports, fits or justifies a service.\n"
        "- No intent words, recommendations or next steps, except the closing 'Records that would clarify this:' sentence. "
        "No headings, lists or bold text.\n"
        "- Never use these words: score, model, anomaly, outlier, z-score, algorithm, fraud, billed.\n"
        "- No intensifiers such as markedly, significantly, dramatically or extremely.\n"
        f"- The first sentence must begin exactly: Provider {b['pid']} was flagged because its {b['cat']} paid claims\n"
        "- 120 to 200 words in 2 or 3 short paragraphs.\n\n"
        "SHOULD (in this order, as the word budget allows):\n"
        "a. Quote the provider value, peer median and percentile for each main pattern that has them.\n"
        "b. Say once how steady the main patterns are (the STABILITY line) and when a pattern was seen at several "
        "analysis levels.\n"
        "c. State the first 'Other explanations' item of main pattern 1 as something that could account for part of the "
        "difference, in one short clause; do not judge whether it applies.\n"
        "d. End with one sentence naming the records that would clarify the main patterns, copied from the RECORDS line, "
        "starting 'Records that would clarify this:'.\n\n"
        + scheme_defs
        + ((EXEMPLAR_MARKER + EXEMPLAR_NOTE + "\n\n") if EXEMPLAR_NOTE else ''))
    user_msg = "Write the case note for this provider.\n\n" + brief_text(b)
    return [{'role': 'system', 'content': system_msg}, {'role': 'user', 'content': user_msg}], bool(b['peers'])


def select_reasoning_level(candidates, n_main, n_levels_agree):
    """Writing a note from a Python brief is not a reasoning task: 'low' for clear cases (a high-evidence candidate),
    NARRATIVE_REASONING['default'] otherwise, never 'high' (v9 used 'high' for empty or diffuse cases; with the v10 prompt
    the model's hidden analysis at 'high' did not finish inside 5,400 tokens on any provider)."""
    if candidates and any(c['evidence'] == 'high' for c in candidates):
        return NARRATIVE_REASONING['clear']
    return NARRATIVE_REASONING['default']


# ── Layer 2: LLM validation of the deterministic candidates against the brief (never against a narrative) ──
def layer2_needed(candidates, n_main, audit_rng):
    """Layer 2 can only confirm or drop deterministic candidates, so it runs when there is something to weigh:
    a low-evidence candidate, or candidates with three main patterns; plus a random audit sample of everything else."""
    if not LAYER2_ENABLED:
        return False
    if candidates and (any(c['evidence'] == 'low' for c in candidates) or n_main >= 3):
        return True
    return audit_rng.random() < LAYER2_AUDIT_RATE


def layer2_menu(cat, candidates, main_rows):
    """Schemes Layer 2 may return: the category menu plus every dictionary link of the main patterns."""
    menu = list(CATEGORY_SCHEME_MENU.get(cat, []))
    for _, r in main_rows.iterrows():
        for s in _links(r):
            if s in SCHEME_REGISTRY and s not in menu:
                menu.append(s)
    for c in candidates:
        if c['scheme'] not in menu:
            menu.append(c['scheme'])
    return menu


def build_layer2_prompt(brief, candidates, menu):
    priors = ("PRE-CLASSIFIED CANDIDATES (from the measured values):\n"
              + ''.join(f"  - {c['scheme']} ({c['evidence']}): {c['rationale']}\n" for c in candidates)) if candidates \
        else "PRE-CLASSIFIED CANDIDATES: none.\n"
    defs = '\n'.join(f"- {s}: {SCHEME_REGISTRY[s]}" for s in menu)
    system_msg = ("You are a Medicaid paid-claims scheme classifier.\n\nSCHEME DEFINITIONS:\n" + defs + "\n\n" + NO_CLINICAL_DATA
                  + "\n\nReturn ONLY pipe-separated lines, one scheme per line: scheme | evidence_strength | one-sentence rationale. "
                  "evidence_strength is high, medium or low. Confirm a pre-classified candidate only when the MAIN PATTERNS and "
                  "their values support it; leave it out to drop it. You may name one other scheme from the list when a MAIN "
                  "pattern clearly supports it (it is recorded for review, not reported as confirmed). Return 'other | low | "
                  "no scheme supported' when nothing is supported. No prose, no table, no header line.")
    user_msg = (f"{priors}\nEVIDENCE BRIEF:\n\n{brief_text(brief, for_layer2=True)}\n\n"
                f"Choose ONLY from: {', '.join(menu)}\nReturn 1 to 3 lines in the exact pipe format.")
    return [{'role': 'system', 'content': system_msg}, {'role': 'user', 'content': user_msg}]


def _scheme_key(text):
    t = re.sub(r'[-/_]', ' ', str(text).lower())
    t = re.sub(r'[^a-z ]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


_SCHEME_KEYS = {_scheme_key(s): s for s in KNOWN_SCHEMES}


def _match_scheme(text):
    """Map a Layer 2 scheme name onto the registry: exact (ignoring case, hyphens, slashes and underscores), then the
    longest registry name contained in the text; a registry name is matched inside a shorter text only when the text
    covers most of it (so 'billing' or 'manipulation' alone match nothing)."""
    t = _scheme_key(text)
    if not t:
        return None
    if t in _SCHEME_KEYS:
        return _SCHEME_KEYS[t]
    if t == 'other':
        return 'other'
    hits = [k for k in _SCHEME_KEYS if f' {k} ' in f' {t} ']
    if hits:
        return _SCHEME_KEYS[max(hits, key=len)]
    hits = [k for k in _SCHEME_KEYS if len(t) >= 0.6 * len(k) and f' {t} ' in f' {k} ']
    return _SCHEME_KEYS[min(hits, key=len)] if hits else None


def parse_layer2(raw, candidates, main_rows, menu):
    """Layer 2 can confirm or drop a deterministic candidate, never add to Detected_Schemes.
    kept:      candidates the answer confirmed (strength never above the deterministic strength)
    dropped:   candidates the answer left out or marked 'other'
    added:     registry schemes the dictionary links to a main pattern but the values did not confirm (for review)
    suggested: menu schemes no main pattern links to (for review of the dictionary links)
    rejected:  names outside the menu and the links (invented)
    usable:    False when no line named a registry scheme or 'other'; the caller then keeps the candidates."""
    linked = {s for _, r in main_rows.iterrows() for s in _links(r)}
    det = {c['scheme']: c for c in candidates}
    rank = {'high': 2, 'medium': 1, 'low': 0}
    out, added, suggested, rejected, usable = [], [], [], [], False
    for line in str(raw or '').strip().splitlines():
        line = line.strip().strip('|').strip()
        if not line or set(line) <= set('-: |') or line.lower().startswith(('scheme |', 'scheme|')):
            continue
        parts = [p.strip() for p in line.split('|')]
        if len(parts) < 2 or not parts[0]:
            continue
        scheme = _match_scheme(parts[0].lstrip('-• ').strip())
        if scheme is None:                      # not a registry name: recorded, but does not make the answer usable
            rejected.append(parts[0].strip()[:40].lower())
            continue
        usable = True
        if scheme == 'other':
            continue
        ev_word = re.sub(r'[^a-z]', ' ', parts[1].lower()).split()
        ev = next((w for w in ev_word if w[:3] in ('hig', 'med', 'low')), None)
        ev = {'hig': 'high', 'med': 'medium', 'low': 'low'}[ev[:3]] if ev else None
        why = parts[2].strip() if len(parts) > 2 else ''
        if scheme in det:
            if scheme in {c['scheme'] for c in out}:
                continue
            if ev is None or rank[ev] > rank[det[scheme]['evidence']]:
                ev = det[scheme]['evidence']
            out.append({'scheme': scheme, 'evidence': ev, 'rationale': why or det[scheme]['rationale'], 'labels': det[scheme]['labels']})
        elif scheme in linked:
            added.append(scheme)
        elif scheme in menu:
            suggested.append(scheme)
        else:
            rejected.append(scheme)
    kept = {c['scheme'] for c in out}
    dropped = [c['scheme'] for c in candidates if c['scheme'] not in kept] if usable else []
    return {'kept': out, 'dropped': dropped, 'added': list(dict.fromkeys(added)), 'suggested': list(dict.fromkeys(suggested)),
            'rejected': list(dict.fromkeys(rejected)), 'usable': usable}


def facts_text(brief):
    """The numbered facts of a brief (provider line, peer lines, value sentences, stability, records, codes): the only
    places a note may take a dollar amount, a percentage or a percentile from."""
    parts = [f"PROVIDER: {brief['pid']}"] + list(brief['peers'])
    for p in brief['main'] + brief['supporting'] + brief['background']:
        parts += [p.get('value_sentence', ''), p.get('solidity', ''), p.get('codes', '')]
    parts += [brief['stability_summary']] + list(brief['records'])
    return '\n'.join(x for x in parts if x)


# ── Deterministic notes (Python only) ──
def evidence_list_note(brief, g, reason):
    """Note for a provider without an LLM narrative: the billing measures (main, then supporting) with their values,
    stability and agreement, the scheme verdict and the records to pull. Background (patient mix, context) stays on the card."""
    pid, cat = brief['pid'], brief['cat']
    items = g[g['Evidence_Rank'].notna()].sort_values('Evidence_Rank') if g is not None else None
    if items is None or items.empty:
        if reason == 'no feature rows':
            return (f"Provider {pid} was flagged because its {cat} paid claims differed from peers. No explaining measures "
                    "were available for this provider in this run.")
        return (f"Provider {pid} was flagged because its {cat} paid claims differed from peers. None of the strongest "
                "measures behind the flag describes a billing pattern that can be narrated yet; all measures are on the evidence card.")
    billing = items[items['Shown_As'].isin(['Main', 'Supporting'])]
    n_bg = int((items['Shown_As'] == 'Background').sum())
    if billing.empty:
        return (f"Provider {pid} was flagged because its {cat} paid claims differed from peers. The strongest measures behind "
                "the flag describe the patient mix or the setting rather than a billing pattern "
                f"({'; '.join(items['Label'].head(3))}); they are listed on the evidence card and are not a reason to open a case on their own.")
    parts = []
    for k, (_, r) in enumerate(billing.iterrows(), 1):
        s = r['Label']
        if r['Code_Decoded']:
            s += f" [{r['Code_Decoded']}]"
        if r['Value_Sentence']:
            s += f": {r['Value_Sentence'].rstrip('.')}"
        else:
            s += ": unusual compared with peers (value not in the claims extract)"
        sol = _solidity(r)
        if sol:
            s += f" ({sol})"
        parts.append(f"({k}) {s}")
    text = f"Provider {pid} was flagged because its {cat} paid claims stood out on: " + '; '.join(parts) + '. '
    sch = brief['schemes']
    if sch['allowed']:
        text += 'The main patterns are consistent with ' + ', '.join(sch['allowed']) + '. '
    elif sch['detected'] == NEW_PATTERN_LABEL:
        text += 'The main patterns match no known scheme (possible new pattern). '
    elif str(sch['detected']).startswith('Not assessed'):
        text += 'No scheme was assessed: none of these measures is a main pattern. '
    elif sch['rationale']:
        text += f"No scheme is confirmed ({sch['rationale']}). "
    else:
        text += 'No scheme is confirmed by the measured values. '
    benign = [p['benign'] for p in brief['main'] if p.get('benign')]
    if benign:
        text += 'Other explanations to rule out: ' + _first_clause(benign[0]).rstrip('.') + '. '
    if n_bg:
        text += (f"{n_bg} context measure{'s' if n_bg > 1 else ''} (patient mix or setting) "
                 f"{'are' if n_bg > 1 else 'is'} on the evidence card. ")
    if brief['records']:
        text += 'Records that would clarify this: ' + '; '.join(brief['records']) + '.'
    if reason == 'generation failed':
        text += ' A written case note could not be produced in this run.'
    return text.strip()


def patient_mix_note(pid, cat, labels):
    lab = f" ({labels.lower()})" if labels else ''
    return (f"Provider {pid} was flagged in {cat} mainly because its patient mix differs from peers{lab}. "
            "Patient mix alone is not a billing concern. None of the strongest measures behind the flag describes how "
            "the provider bills, so this provider is listed for review of the detection method. All measures are on the evidence card.")

##### Generation

In [ ]:
# ── Generation with GPT-OSS (Harmony channels; the hidden reasoning is never published) ──
def _harmony_text(content):
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, (list, tuple)):
        return ''.join(_harmony_text(c) for c in content)
    text = getattr(content, 'text', None)
    return text if isinstance(text, str) else ''


def _parse_completion(new_token_ids, max_new_tokens):
    """(final_text, analysis_chars, meta) from generated token ids."""
    meta = {'new_tokens': len(new_token_ids), 'hit_token_limit': len(new_token_ids) >= max_new_tokens}
    final_text, analysis = '', ''
    try:
        from openai_harmony import Role
        for msg in harmony_enc.parse_messages_from_completion_tokens(new_token_ids, Role.ASSISTANT):
            channel = str(getattr(msg, 'channel', '') or '')
            text = _harmony_text(getattr(msg, 'content', None))
            if channel == 'final':
                final_text = text
            elif channel == 'analysis':
                analysis += text
        meta['parse'] = 'harmony'
    except Exception as e:
        meta['parse'] = f'fallback ({type(e).__name__})'
        raw = tokenizer.decode(new_token_ids, skip_special_tokens=False)
        if '<|channel|>final<|message|>' in raw:
            final_text = raw.split('<|channel|>final<|message|>')[-1]
            for stop in ['<|return|>', '<|end|>', '<|start|>']:
                final_text = final_text.split(stop)[0]
    meta['final_found'] = bool(final_text.strip())
    return final_text.strip(), len(analysis), meta


def _stop_token_ids():
    try:
        return set(harmony_enc.stop_tokens_for_assistant_actions())
    except Exception:
        return set()


_LEAK_SIGNATURES = ('analysisWe need', 'We need to write', '<|channel|>', 'assistantfinal')


def looks_like_reasoning_leak(text):
    """v9: hidden reasoning that reached the final text (never published)."""
    head = str(text or '')[:400]
    return bool(head) and (re.match(r'^\s*analysis', head) is not None or any(s in head for s in _LEAK_SIGNATURES))


class GenerationCircuitBreaker:
    """v9: stop the run when generation keeps failing (a broken model is not a provider problem)."""
    def __init__(self):
        self.consecutive = self.failures = self.total = 0

    def record(self, failed):
        self.total += 1
        self.failures += int(failed)
        self.consecutive = self.consecutive + 1 if failed else 0
        if self.consecutive >= BREAKER_MAX_CONSECUTIVE:
            raise RuntimeError(f"CIRCUIT BREAKER: {self.consecutive} consecutive generation failures; fix the generator before continuing.")
        if self.total >= BREAKER_MIN_SAMPLE and self.failures / self.total > BREAKER_MAX_RATE:
            raise RuntimeError(f"CIRCUIT BREAKER: failure rate {self.failures}/{self.total} above {BREAKER_MAX_RATE:.0%}.")


def generate_batch(message_lists, max_new_tokens, reasoning_level='medium'):
    """One model.generate call for a list of conversations. Returns [(final_text, analysis_chars, meta)]."""
    if len(message_lists) == 1:
        inputs = tokenizer.apply_chat_template(message_lists[0], return_tensors='pt', add_generation_prompt=True,
                                               return_dict=True, reasoning_effort=reasoning_level)
        input_ids = inputs['input_ids'].to(model.device)
        attention_mask = inputs.get('attention_mask')
        attention_mask = attention_mask.to(model.device) if attention_mask is not None else None
        with torch.no_grad():
            out = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, do_sample=False)
        return [_parse_completion(out[0][input_ids.shape[1]:].tolist(), max_new_tokens)]
    # Batched: left padding so every prompt ends at the same position
    seqs = []
    for msgs in message_lists:
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True,
                                            reasoning_effort=reasoning_level)['input_ids']
        seqs.append(list(ids[0]) if ids and isinstance(ids[0], (list, tuple)) else list(ids))
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    width = max(len(s) for s in seqs)
    input_ids = torch.tensor([[pad] * (width - len(s)) + s for s in seqs]).to(model.device)
    attention_mask = torch.tensor([[0] * (width - len(s)) + [1] * len(s) for s in seqs]).to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=pad)
    stops, results = _stop_token_ids(), []
    for row in out:
        new = row[width:].tolist()
        cut = next((i for i, t in enumerate(new) if t in stops), None)
        if cut is not None:
            new = new[:cut + 1]
        results.append(_parse_completion(new, max_new_tokens))
    return results


def _is_cut_off(text, meta):
    return meta.get('hit_token_limit') and not normalize_text(text).rstrip('"\')').endswith(('.', '!', '?'))


def write_notes(jobs):
    """jobs: list of dicts with 'messages'. Adds raw_final, attempts, new_tokens, hit_token_limit, parse, error.
    One retry when the answer is missing or cut off: at RETRY_REASONING effort (the hidden analysis is what overflows a
    budget; more tokens for an analysis that did not converge is the wrong lever) with RETRY_FACTOR times the budget of the
    job's own level; on a missing answer the retry also asks for the final note directly."""
    pending = list(jobs)
    for attempt in (1, 2):
        if not pending:
            break
        retry = []
        if GEN_BATCH_SIZE > 1:      # jobs in one GPU call share a reasoning level and budget; otherwise keep provider order
            pending.sort(key=lambda j: j.get('reasoning_level', 'medium'))
        for i in range(0, len(pending), max(GEN_BATCH_SIZE, 1)):
            chunk = pending[i:i + max(GEN_BATCH_SIZE, 1)]
            level = chunk[0].get('reasoning_level', 'medium')
            budget = int(REASONING_TOKEN_BUDGET.get(level, 1400) * (RETRY_FACTOR if attempt > 1 else 1))
            if attempt > 1:
                level = RETRY_REASONING
            msgs = []
            for j in chunk:
                m = j['messages']
                if attempt > 1 and not j.get('hit_token_limit'):
                    m = m[:-1] + [{'role': 'user', 'content': m[-1]['content']
                                   + j.get('retry_suffix', "\nWrite only the final case note now.")}]
                msgs.append(m)
            try:
                results = generate_batch(msgs, budget, level)
            except Exception as e:
                if len(chunk) > 1:          # e.g. out of memory: fall back to one at a time
                    torch.cuda.empty_cache() if torch.cuda.is_available() else None
                    results = []
                    for m in msgs:
                        try:
                            results.extend(generate_batch([m], budget, level))
                        except Exception as e1:
                            results.append(('', 0, {'error': f'{type(e1).__name__}: {e1}'[:300]}))
                else:
                    results = [('', 0, {'error': f'{type(e).__name__}: {e}'[:300]})]
            for j, (text, a_chars, meta) in zip(chunk, results):
                if looks_like_reasoning_leak(text):
                    meta = dict(meta, error='reasoning leak in final text', hit_token_limit=False)
                    text = ''
                j.update(raw_final=text, attempts=attempt, analysis_chars=a_chars, effort_used=level,
                         new_tokens=meta.get('new_tokens', 0), hit_token_limit=bool(meta.get('hit_token_limit')),
                         parse=meta.get('parse', ''), error=meta.get('error', ''))
                if attempt == 1:
                    j['first_attempt_hit_limit'] = bool(meta.get('hit_token_limit'))
                if (not text or _is_cut_off(text, meta)) and attempt == 1:
                    retry.append(j)
        pending = retry
    return jobs


_SENT = re.compile(r'(?<=[.!?])(?<!\bvs\.)(?<!e\.g\.)(?<!i\.e\.)(?<!approx\.)\s+(?=[A-Z"(])')


def clean_note(text):
    """Normalize characters; drop markdown, headings and repeated sentences; keep paragraphs."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = normalize_text(text)
    text = re.sub(r'<\|[^|]+\|>', '', text)
    text = re.sub(r'(?m)^[ \t]{0,3}(#{1,6}[ \t]*|[-*•][ \t]+|\d+[.)][ \t]+)', '', text)
    text = text.replace('**', '').replace('__', '')
    text = re.sub(r'(?im)^[ \t]*(summary|case note|answer)[ \t]*:[ \t]*', '', text)
    paras, seen = [], set()
    for para in re.split(r'\n\s*\n', text):
        kept = []
        for s in _SENT.split(para.strip()):
            key = s.strip().lower().rstrip('.')
            if key and key not in seen:
                seen.add(key)
                kept.append(s.strip())
        if kept:
            paras.append(' '.join(kept))
    return re.sub(r'[ \t]+', ' ', '\n\n'.join(paras)).strip()

##### Quality checks

In [ ]:
# ── QA: three checks (numbers grounded in the prompt, banned words, opening and category) plus length ──
QA_BANNED = [
    (r'\bscor(e|es|ed|ing)\b', 'score'), (r'\bz-?scores?\b', 'z-score'), (r'\banomal', 'anomaly'),
    (r'\boutliers?\b', 'outlier'), (r'\bmodel(s|ed|ing)?\b', 'model'), (r'\bshap\b', 'shap'),
    (r'\balgorithm', 'algorithm'), (r'machine learning|autoencoder|isolation forest', 'ml term'),
    (r'\bbilled\b', 'billed'), (r'\bfraud', 'fraud'),
    (r'\b(markedly|significantly|dramatically|substantially|extremely|grossly|alarmingly|egregious(ly)?)\b', 'intensifier'),
    (r'\b(deliberate(ly)?|intentional(ly)?|knowingly|purpose(ly|fully)|to maximi[sz]e|in order to (increase|inflate|obtain)'
     r'|gam(e|ed|ing) the|exploit(s|ed|ing)?)\b', 'intent'),
    (r'medical(ly)? necess|clinically (justif|appropriate|necessary|indicated)|\bunjustified\b|\bunnecessary\b'
     r'|\b(does|did|do) not (support|fit|match|justify)\b|\binconsistent with (the |a |its )?diagnos'
     r'|\bnot (supported|indicated|justified) by\b|clinically (unwarranted|unnecessary|inappropriate|unsupported)'
     r'|\bwould not (ordinarily |normally |usually )?(require|need)|\bnot (medically )?(required|warranted)\b', 'clinical judgment'),
    (r'\b(was|were|been) (not|never) (actually )?(delivered|provided|rendered|performed|received)\b'
     r'|\bnot (actually )?(delivered|provided|rendered|received|performed)\b|\bnever (happened|occurred|took place)\b'
     r'|\bdid not (actually )?(happen|occur|take place)\b|\b(fake|fictitious|phony|bogus|fabricated)\b'
     r'|\bmay (not|never) have (taken place|occurred|happened)\b|\bno (evidence|record|sign) (that )?(the )?\w+ (occurred|took place|happened)\b',
     'delivery claim'),
    (r'\brecommend|should (be )?(investigat|review|audit)|\bwarrants?\b|further (review|investigation)|next steps?\b',
     'recommendation'),
    (r'\btwice (as|the)\b|\b(double|triple|quadruple) (the|that)\b|\b\d+(\.\d+)?\s*(times|-fold|x)\s+(the|as|higher|more|that|what|peers?)\b'
     r'|\b(two|three|four|five|six|ten) times\b|\b(half|a third|one third|a quarter|two thirds) (of |as |the )',
     'ratio'),
]
# SCHEME_SYNONYMS is defined in the configuration cell (shared with the brief filter)
_NUM_RE = re.compile(r'(?<![A-Za-z0-9_.])-?\$?\d[\d,]*(?:\.\d+)?(?:%|st|nd|rd|th)?(?![A-Za-z0-9_])')


def _numbers(text):
    out = []
    for m in _NUM_RE.finditer(normalize_text(text)):
        tok = m.group(0)
        core = re.sub(r'[^\d.\-]', '', tok.replace(',', '')).rstrip('.')
        try:
            val = float(core)
        except ValueError:
            continue
        after = normalize_text(text)[m.end():m.end() + 12].lower()
        pct_word = after.startswith(' percent') and not after.startswith(' percentil')
        kind = '$' if '$' in tok else '%' if (tok.endswith('%') or pct_word) else 'ord' if re.search(r'(st|nd|rd|th)$', tok) else 'n'
        out.append((tok, val, kind))
    return out


NECESSITY_ASSERTIONS = ['was not medically necessary', 'were not medically necessary', 'is not medically necessary',
                        'medically inappropriate', 'clinically inappropriate', 'no medical need', 'medical records show',
                        'clinical documentation shows']


_LABEL_STOP = {'paid', 'lines', 'line', 'share', 'with', 'from', 'this', 'that', 'provider', 'patient', 'patients', 'claims', 'claim'}


def _label_words(label):
    """Words that identify a measure in a sentence; falls back to every word of 3+ letters when the label is all stopwords
    ('Paid claim lines per patient')."""
    words = re.sub(r'[^a-z0-9 ]+', ' ', str(label).lower()).split()
    out = {w for w in words if len(w) > 3 and w not in _LABEL_STOP}
    return out or {w for w in words if len(w) > 2}


def quality_check(note, pid, cat, prompt_user, allowed_schemes, value_map=None, facts=None, main_first=None, prompt_system=''):
    """value_map: {label: [numbers]} for the main and supporting patterns; a quoted number must sit in a sentence that
    names its own measure (same-sentence rule), so values cannot migrate between measures.
    facts: the numbered fact lines of the brief; dollar amounts, percentages and percentiles must come from there
    (plain counts may also come from descriptions and policy limits elsewhere in the prompt).
    main_first: (label, first provider value or None) of main pattern 1; the note must name it, and quote its value.
    prompt_system: the system message; its rules (not the exemplar) are wording the note may legitimately repeat."""
    issues, s = [], normalize_text(note)
    words = len(s.split())
    if words < 40:
        issues.append('too_short')
    if words > 260:
        issues.append('too_long')
    if s and not s.rstrip().rstrip('"\')').endswith(('.', '!', '?')):
        issues.append('truncated')
    # (1) Banned words, after removing the scheme names the note may use
    low = s.lower()
    for label in sorted([k for k in KNOWN_SCHEMES if k != 'other'], key=len, reverse=True):
        if label in low:
            if label not in allowed_schemes:
                issues.append(f'unsupported_scheme:{label}')
            low = low.replace(label, ' ')
    for pat, label in SCHEME_SYNONYMS:
        m = re.search(pat, low)
        if m and label not in allowed_schemes:
            issues.append(f'unsupported_scheme:{label}({m.group(0)})')
        low = re.sub(pat, ' ', low)
    for label in allowed_schemes:            # an allowed scheme may only be named as 'consistent with'
        for m in re.finditer(re.escape(label), s.lower()):
            if 'consistent with' not in s.lower()[max(0, m.start() - 45):m.start()]:
                issues.append(f'scheme_as_fact:{label}')
                break
    _prompt_low = normalize_text(prompt_user).lower()
    for pat, name in QA_BANNED:
        m = re.search(pat, low)
        if m and not (name == 'ratio' and m.group(0) in _prompt_low):    # a ratio phrase copied from the brief is Python's, not the model's
            issues.append(f'banned:{name}({m.group(0)})')
    # (2) Every number must appear in the prompt as the same kind ($, %, ordinal, plain); list numbering ignored
    p_text = re.sub(r'(?m)^\s*\d+\.\s', ' ', normalize_text(prompt_user))
    p_items = [(v, k) for _, v, k in _numbers(p_text)]

    f_items = [(v, k) for _, v, k in _numbers(normalize_text(facts))] if facts else None

    def _grounded(v, k):     # exact, or a prompt value of 10 or more rounded half up to a whole number
        pool = f_items if (f_items is not None and k in ('$', '%', 'ord')) else p_items
        return any(k == pk and (v == pv or (abs(pv) >= 10 and v == math.floor(pv + 0.5))) for pv, pk in pool)
    bad = sorted({tok for tok, v, k in _numbers(s) if not _grounded(v, k)})
    if bad:
        issues.append(f"ungrounded_number:{'|'.join(bad[:5])}")
    # A scheme definition (system message only) repeated as if it were a finding about this provider
    user_words = _norm_words(prompt_user)
    for label in allowed_schemes:
        w = re.sub(r'[^a-z0-9 ]+', ' ', SCHEME_DEFINITIONS_BY_LABEL.get(label, '').lower()).split()
        for i in range(len(w) - 5):
            frag = ' ' + ' '.join(w[i:i + 6]) + ' '
            if frag in _norm_words(s) and frag not in user_words:
                issues.append(f'definition_as_finding:{label}')
                break
    # The exemplar (system message) copied into the note: 7-word fragments of it that are neither in the brief nor in the
    # rules (template phrases such as 'Records that would clarify this' and the stability wording are expected to repeat)
    if EXEMPLAR_NOTE:
        rules = str(prompt_system or '').split(EXEMPLAR_MARKER)[0]
        allowed_words = user_words + ' ' + _norm_words(rules) + ' ' + _norm_words(f"Provider {pid} was flagged because its {cat} paid claims")
        stab_words = set(' '.join(STABILITY_PHRASE.values()).split()) | {'this', 'pattern', 'was', 'were', 'and', 'the'}
        w = re.sub(r'[^a-z0-9 ]+', ' ', EXEMPLAR_NOTE.lower()).split()
        note_words = _norm_words(s)
        for i in range(len(w) - 6):
            frag_words = w[i:i + 7]
            if len([x for x in frag_words if x not in stab_words]) < 5:      # mostly the stability wording: expected to repeat
                continue
            frag = ' ' + ' '.join(frag_words) + ' '
            if frag in note_words and frag not in allowed_words:
                issues.append('exemplar_copied')
                break
    # Provider value and peer median must not trade places ('provider value (peer median X' pairs from the prompt)
    for pv, pmed in re.findall(r'(\$?[\d,]+(?:\.\d+)?%?)[^()\n]{0,80}\(peer median (\$?[\d,]+(?:\.\d+)?%?)', normalize_text(prompt_user)):
        if pv != pmed and (re.search(rf'median (?:of )?(?:about |around )?{re.escape(pv)}(?![\d,])', s)
                           or re.search(rf'{re.escape(pmed)}(?![\d,])[^.]{{0,40}}against a peer median', s)):
            issues.append(f'swapped_value:{pv}/{pmed}')
    # (2b) Same-sentence rule: a number that belongs to one measure must appear in a sentence naming that measure
    if value_map:
        owners = {}
        for label, nums in value_map.items():
            for n in nums:
                owners.setdefault(n, set()).add(label)
        for sent in _SENT.split(s):
            sent_norm = _norm_words(sent)
            for tok, v, k in _numbers(sent):
                if v in owners and (k in ('$', '%', 'n')) and v >= 10:
                    if not any(_label_words(lab) & set(sent_norm.split()) for lab in owners[v]):
                        issues.append(f'number_away_from_measure:{tok}')
                        break
    for term in NECESSITY_ASSERTIONS:
        if term in s.lower():
            issues.append(f'necessity_assertion:{term}')
            break
    # (2c) The note must be about main pattern 1: name it, and quote its provider value when there is one
    if main_first:
        lab, val = main_first
        if not (_label_words(lab) & set(_norm_words(s).split())):
            issues.append('main_pattern_missing')
        elif val is not None and not any(v == val for _, v, _ in _numbers(s)):
            issues.append('main_value_missing')
    # (3) Opening sentence and category
    first = (_SENT.split(s.strip()) or [''])[0]
    if not re.match(rf'\s*Provider\s+{re.escape(str(pid))}\s+was flagged because its\b', first):
        issues.append('opening_template')
    if not category_mentioned(first, cat):
        issues.append('category_not_in_opening')
    return issues or ['pass']

##### Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai_harmony import load_harmony_encoding, HarmonyEncodingName

_n_llm = int(providers['Narrate_Selected'].sum())
if _n_llm == 0:
    print("No provider selected for a narrative; the model is not loaded.")
    tokenizer = model = harmony_enc = None
else:
    _hf_token = os.environ.get('HF_TOKEN')
    if not _hf_token and HF_SECRET_SCOPE:
        _hf_token = dbutils.secrets.get(scope=HF_SECRET_SCOPE, key=HF_SECRET_KEY)
    if _hf_token:
        login(_hf_token)
    print(f"Loading {MODEL_ID} for {_n_llm} narratives")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if GEN_BATCH_SIZE > 1:
        tokenizer.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto", trust_remote_code=True)
    harmony_enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
    print(f"Model loaded; device map: {getattr(model, 'hf_device_map', 'auto')}")

##### Write the notes
Python notes first; Layer 2 validation; narratives with checkpoint, resume and circuit breaker.

In [ ]:
# ── Write the notes: Layer 1 brief (Python) -> Layer 2 scheme validation (LLM, when needed) -> Layer 3 narrative (LLM) ──
import random
audit_rng = random.Random(LAYER2_SEED)


def _value_map(g):
    """{label: [numbers]} of the main and supporting patterns, for the same-sentence QA rule."""
    out = {}
    for _, r in g[g['Shown_As'].isin(['Main', 'Supporting'])].iterrows():
        nums = [v for _, v, _ in _numbers(r['Value_Sentence'])] if r['Value_Sentence'] else []
        if nums:
            out[r['Label']] = nums
    return out


def _main_first(g):
    """(label, first provider value or None) of main pattern 1, for the QA coverage check."""
    main = g[g['Shown_As'] == 'Main'].sort_values('Feature_Rank_The_Provider')
    if main.empty:
        return None
    r = main.iloc[0]
    nums = [v for _, v, _ in _numbers(r['Value_Sentence'])] if r['Value_Sentence'] else []
    return (r['Label'], nums[0] if nums else None)


def _qa(note, j):
    g = features_by_provider[j['key']]
    return quality_check(note, j['row'][provider_id], j['row']['Category'], j['messages'][1]['content'], j['allowed'],
                         _value_map(g), facts_text(j['brief']), _main_first(g), j['messages'][0]['content'])


def _base_row(p, schemes):
    key = p['_key']
    g = features_by_provider.get(key)
    main = g[g['Shown_As'] == 'Main'] if g is not None else None
    return {
        provider_id: p[provider_id], 'Category': p['Category'], 'LLM_Summary': '',
        'Detected_Schemes': schemes['detected'], 'Scheme_Rationale': schemes['rationale'], 'Possible_Schemes': schemes['possible'],
        'Scheme_Strength': p['Scheme_Strength'], 'Layer2_Used': False, 'Layer2_Dropped': '', 'Layer2_Added': '',
        'Layer2_Suggested': '', 'Layer2_Rejected': '', 'Layer2_Note': '',
        'QA_Flags': '', 'QA_Status': '', 'Baselines_Available': False, 'Scoring_Date': scoring_date,
        'Narrative_Features': ' | '.join(main['Label']) if main is not None else '',
        'Main_Pattern_Confidence': (min(main['Definition_Confidence'], key=CONFIDENCE_ORDER.index) if main is not None and len(main) else ''),
        'Main_Pattern_Stability': ' | '.join(main['Stability'].astype(str)) if main is not None else '',
        'Notebook_Version': NOTEBOOK_VERSION, 'Dictionary_Version': DICTIONARY_VERSION,
        'Note_Type': p['Note_Type'], 'Note_Reason': p['Note_Reason'], 'Reasoning_Level': '',
        'Provider_Anomaly_Score': p['Servicing_Provider_Anomaly_Score'], 'Category_Rank': p['Category_Rank'],
        'Narrate_Selected': bool(p['Narrate_Selected']), 'Patient_Mix_Share': p.get('Patient_Mix_Share'),
        'Provider_Perspective_Score': p['Provider_Perspective_Score'], 'Patient_Perspective_Score': p['Patient_Perspective_Score'],
        'Visit_Perspective_Score': p['Visit_Perspective_Score'], 'Claim_Perspective_Score': p['Claim_Perspective_Score'],
    }


def _prompt_sha(messages):
    return hashlib.sha1('\n'.join(m['content'] for m in messages).encode()).hexdigest()


# Resume: a narrative is reused only when it was written for exactly the same prompt and did not fail
done, _bad_lines = {}, 0
if RESUME and os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                _bad_lines += 1
                continue
            if rec['summary'].get('QA_Status') != 'generation_failed':
                done[(str(rec['summary'][provider_id]), rec['summary']['Category'])] = rec
    print(f"Checkpoint: {len(done)} narratives available for reuse" + (f"; {_bad_lines} unreadable lines skipped" if _bad_lines else ''))

summary_rows, audit_rows, jobs, reused = [], [], [], 0
briefs = {}
breaker = GenerationCircuitBreaker()
for p in providers.to_dict('records'):
    key = p['_key']
    schemes = dict(_sb[key])
    g = features_by_provider.get(key)
    row = _base_row(p, schemes)
    if p['Note_Type'] == 'patient_mix':
        row.update(LLM_Summary=patient_mix_note(p[provider_id], p['Category'], p.get('Mix_Labels', '')),
                   QA_Flags='patient_mix', QA_Status='patient_mix')
        summary_rows.append(row)
        continue
    brief = build_brief(p[provider_id], p['Category'], g if g is not None else feat.iloc[0:0], schemes)
    briefs[key] = brief
    if p['Note_Type'] in ('evidence_only', 'evidence_list'):
        row.update(LLM_Summary=evidence_list_note(brief, g, p['Note_Reason']), QA_Flags=p['Note_Type'], QA_Status='evidence_only')
        summary_rows.append(row)
        continue
    # narrative: Layer 2 first (on the brief), then the narrative prompt with the validated schemes
    main = g[g['Shown_As'] == 'Main']
    n_agree = int(main['N_Levels'].max()) if len(main) else 1
    level = select_reasoning_level(schemes['candidates'], len(main), n_agree)
    row['Reasoning_Level'] = level
    jobs.append({'row': row, 'brief': brief, 'main': main, 'schemes': schemes, 'key': key, 'provider': p,
                 'reasoning_level': level, 'layer2': layer2_needed(schemes['candidates'], len(main), audit_rng)})

print(f"Deterministic notes: {len(summary_rows)} | narratives to produce: {len(jobs)} | "
      f"Layer 2 validation on {sum(j['layer2'] for j in jobs)} of them")

# ── Layer 2 (LLM) for the jobs that need it ──
_l2 = [j for j in jobs if j['layer2']]
if _l2:
    for j in _l2:
        j['l2_menu'] = layer2_menu(j['provider']['Category'], j['schemes']['candidates'], j['main'])
        j['l2_messages'] = build_layer2_prompt(j['brief'], j['schemes']['candidates'], j['l2_menu'])
    _l2_jobs = [{'messages': j['l2_messages'], 'reasoning_level': 'low',
                 'retry_suffix': "\nReturn only the pipe-separated lines now."} for j in _l2]
    _saved = REASONING_TOKEN_BUDGET.copy()
    REASONING_TOKEN_BUDGET['low'] = LAYER2_MAX_NEW_TOKENS
    try:
        write_notes(_l2_jobs)
    finally:
        REASONING_TOKEN_BUDGET.update(_saved)
    for j, r in zip(_l2, _l2_jobs):
        res = parse_layer2(r.get('raw_final', ''), j['schemes']['candidates'], j['main'], j['l2_menu'])
        if r.get('raw_final') and res['usable']:
            kept = res['kept']
            sch = j['schemes']
            sch['candidates'] = kept
            sch['allowed'] = [c['scheme'] for c in kept if c['scheme'] in SCHEME_DEFINITIONS_BY_LABEL]
            possible = [x.strip() for x in str(sch['possible']).split(',') if x.strip()]
            sch['possible'] = ', '.join(dict.fromkeys(possible + res['dropped'] + res['added']))
            sch['detected'] = (', '.join(f"{c['scheme']} ({c['evidence']} evidence)" for c in kept) if kept
                               else ('Not confirmed (see Possible_Schemes)' if sch['possible'] else sch['detected']))
            strength = min((c['evidence'] for c in kept), key=['high', 'medium', 'low'].index) if kept else ''
            j['row'].update(Detected_Schemes=sch['detected'], Possible_Schemes=sch['possible'], Scheme_Strength=strength,
                            Layer2_Used=True, Layer2_Dropped=', '.join(res['dropped']), Layer2_Added=', '.join(res['added']),
                            Layer2_Suggested=', '.join(res['suggested']), Layer2_Rejected=', '.join(res['rejected']))
            # the brief's benign/caveat sentences were filtered with the old allowed set: rebuild it
            j['brief'] = build_brief(j['provider'][provider_id], j['provider']['Category'], features_by_provider[j['key']], sch)
            briefs[j['key']] = j['brief']
        else:
            j['row'].update(Layer2_Used=False, Layer2_Note='Layer 2 gave no usable answer; deterministic schemes kept'
                            + (f" | error: {r.get('error')}" if r.get('error') else ''))

# ── Layer 3: narrative prompts, with resume ──
for j in jobs:
    messages, has_peers = build_prompt(j['brief'], j['schemes']['allowed'], j['reasoning_level'])
    j['messages'], j['allowed'] = messages, j['schemes']['allowed']
    j['row']['Baselines_Available'] = has_peers
to_generate = []
for j in jobs:
    prev = done.get(j['key'])
    if prev is not None and prev['audit'].get('Prompt_SHA1') == _prompt_sha(j['messages']):
        flags = _qa(prev['summary']['LLM_Summary'], j)
        j['row'].update(LLM_Summary=prev['summary']['LLM_Summary'], QA_Flags=', '.join(flags),
                        QA_Status='pass' if flags == ['pass'] else 'needs_review')
        summary_rows.append(j['row'])
        audit_rows.append(prev['audit'])
        reused += 1
    else:
        to_generate.append(j)
print(f"Reused from checkpoint: {reused} | to generate: {len(to_generate)}")

_chunk = max(GEN_BATCH_SIZE, 1) * 5
for start in range(0, len(to_generate), _chunk):
    batch = write_notes(to_generate[start:start + _chunk])
    with open(CHECKPOINT_FILE, 'a') as ckpt:
        for j in batch:
            row, pid, cat = j['row'], j['row'][provider_id], j['row']['Category']
            failed = not j.get('raw_final')
            if not failed:
                note = clean_note(j['raw_final'])
                flags = _qa(note, j)
                row.update(LLM_Summary=note, QA_Flags=', '.join(flags), QA_Status='pass' if flags == ['pass'] else 'needs_review')
            else:
                row.update(LLM_Summary=evidence_list_note(j['brief'], features_by_provider.get(j['key']), 'generation failed'),
                           QA_Flags='generation_failed' + (f": {j['error']}" if j.get('error') else ''),
                           QA_Status='generation_failed', Note_Type='generation_failed')
            breaker.record(failed)
            audit = {provider_id: pid, 'Category': cat, 'Scoring_Date': scoring_date, 'Note_Type': row['Note_Type'],
                     'Reasoning_Level': j['reasoning_level'], 'Effort_Used': j.get('effort_used', ''), 'Layer2_Used': row['Layer2_Used'],
                     'Attempts': j.get('attempts', 0), 'New_Tokens': j.get('new_tokens', 0),
                     'Hit_Token_Limit': j.get('hit_token_limit', False), 'First_Attempt_Hit_Limit': j.get('first_attempt_hit_limit', False),
                     'Parse': j.get('parse', ''), 'Error': j.get('error', ''),
                     'Analysis_Chars': j.get('analysis_chars', 0), 'Prompt_System': j['messages'][0]['content'],
                     'Prompt_User': j['messages'][1]['content'], 'Layer2_Prompt': j.get('l2_messages', [{}, {}])[1].get('content', ''),
                     'Raw_Final': j.get('raw_final', ''), 'Prompt_SHA1': _prompt_sha(j['messages'])}
            summary_rows.append(row)
            audit_rows.append(audit)
            ckpt.write(json.dumps({'summary': row, 'audit': audit}, default=str) + '\n')
    print(f"  generated {min(start + _chunk, len(to_generate))}/{len(to_generate)}")

output_df = pd.DataFrame(summary_rows)
audit_df = pd.DataFrame(audit_rows)
print("\nNotes:", output_df['QA_Status'].value_counts().to_dict())
_narr = output_df[output_df['QA_Status'].isin(['pass', 'needs_review'])]
if len(_narr):
    print(f"Narratives passing QA: {(_narr['QA_Status'] == 'pass').mean():.0%} of {len(_narr)}")

##### Save (client file, quarantine, schemes table)

In [ ]:
# ── Save: client file (QA pass plus Python notes), quarantine (needs_review and failed), schemes table, features, audit ──
MOJIBAKE = {'â€¯': ' ', 'â€"': '-', 'â€“': '-', 'â€œ': '"', 'â€\x9d': '"', 'â€™': "'", 'â€˜': "'", 'Â ': ' ', 'Â': ''}


def scrub_for_excel(text):
    if not isinstance(text, str):
        return text
    for bad, good in MOJIBAKE.items():
        text = text.replace(bad, good)
    return normalize_text(text)


for c in ['LLM_Summary', 'Detected_Schemes', 'Scheme_Rationale', 'Possible_Schemes']:
    output_df[c] = output_df[c].map(scrub_for_excel)
_client_mask = output_df['QA_Status'].isin(['pass', 'evidence_only', 'patient_mix'])
quarantine_df = output_df[~_client_mask].copy()
# A quarantined narrative still leaves its provider with a Python evidence note in the client file (same facts, no LLM text)
_fallback = []
for _, r in quarantine_df.iterrows():
    _key = (r[provider_id], r['Category'])
    if _key in briefs:
        fb = r.copy()
        fb['LLM_Summary'] = scrub_for_excel(evidence_list_note(briefs[_key], features_by_provider.get(_key),
                                                               'narrative quarantined' if r['QA_Status'] == 'needs_review' else 'generation failed'))
        fb['Note_Type'], fb['QA_Status'] = 'evidence_fallback', 'evidence_fallback'
        fb['Note_Reason'] = f"narrative {r['QA_Status']}: {r['QA_Flags']}"[:300]
        _fallback.append(fb)
client_df = pd.concat([output_df[_client_mask], pd.DataFrame(_fallback)], ignore_index=True) if _fallback else output_df[_client_mask].copy()
client_df = client_df.sort_values(['Category', 'Category_Rank'], kind='stable').reset_index(drop=True)
print(f"Client rows: {len(client_df)} (of which evidence fallbacks for quarantined narratives: {len(_fallback)}) | "
      f"quarantined (needs_review, generation_failed): {len(quarantine_df)}")

# Same summary file names as before, so downstream reports keep working; new columns at the end
output_df.to_csv(f'{OUTPUT_DIR}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony.csv', index=False)
output_df.to_csv(f'{OUTPUT_DIR}/all_provider_pretrained_trained_aggregated_llm_based_summary_v3_0_harmony_{scoring_date}.csv', index=False)
client_df.to_csv(f'{OUTPUT_DIR}/all_providers_llm_based_summary_{scoring_date}.csv', index=False, encoding='utf-8-sig')
quarantine_df.to_csv(f'{OUTPUT_DIR}/llm_summary_quarantine_{scoring_date}.csv', index=False, encoding='utf-8-sig')
audit_df.to_json(f'{OUTPUT_DIR}/llm_summary_audit_{scoring_date}.jsonl', orient='records', lines=True)

# Schemes table: one row per confirmed scheme per provider (client rows only)
_sch = []
for _, r in client_df.iterrows():
    for m in re.finditer(r'([a-z][a-z /\-]+?) \((high|medium|low) evidence\)', str(r['Detected_Schemes'])):
        _sch.append({provider_id: r[provider_id], 'Category': r['Category'], 'Scoring_Date': scoring_date,
                     'Detected_Scheme': m.group(1).strip(), 'Evidence_Strength': m.group(2)})
schemes_df = pd.DataFrame(_sch, columns=[provider_id, 'Category', 'Scoring_Date', 'Detected_Scheme', 'Evidence_Strength'])
schemes_df.to_csv(f'{OUTPUT_DIR}/all_providers_llm_based_summary_schemes_{scoring_date}.csv', index=False, encoding='utf-8-sig')

# Evidence card input: every feature of every referred provider, with what the note did with it
features_out = feat[[provider_id, 'Category', 'Feature', 'Feature_Root', 'Label', 'Feature_Category', 'Best_Level',
                     'Levels_In_File', 'N_Levels', 'Stability', 'Consensus', 'Importance_Tier', 'In_Main_Tier',
                     'Feature_Rank_The_Provider', 'Overall_Importance_Score', 'Narrate_Role', 'Validation_Status',
                     'Role', 'Role_Reason', 'Shown_As', 'Evidence_Rank', 'Value_Sentence', 'value', 'peer_median', 'peer_pct',
                     'Scheme_Link', 'Code_Decoded', 'Records_To_Review', 'Policy_Lever', 'Definition_Confidence', 'Level_Dependent']]
features_out.to_csv(f'{OUTPUT_DIR}/llm_summary_features_{scoring_date}.csv', index=False)

_pm = providers[providers['Note_Type'] == 'patient_mix'][[provider_id, 'Category', 'Servicing_Provider_Anomaly_Score',
                                                          'Patient_Mix_Share', 'Mix_Labels', 'Patient_Perspective_Score',
                                                          'Provider_Perspective_Score', 'Visit_Perspective_Score', 'Claim_Perspective_Score']]
_pm.to_csv(f'{OUTPUT_DIR}/patient_mix_providers_{scoring_date}.csv', index=False)
print(f"Saved {len(output_df)} notes ({len(schemes_df)} scheme rows), {len(audit_df)} audit records, "
      f"{len(features_out)} feature rows, {len(_pm)} patient-mix providers.")

##### Save to Delta table

In [ ]:
# Delta write (uncomment to run). Columns are selected by name; mergeSchema adds the new columns to an older table.
# spark = SparkSession.builder.getOrCreate()
# _cols = [provider_id, 'Category', 'LLM_Summary', 'Detected_Schemes', 'Scheme_Rationale', 'QA_Flags', 'QA_Status',
#          'Baselines_Available', 'Scoring_Date', 'Narrative_Features', 'Notebook_Version', 'Dictionary_Version',
#          'Possible_Schemes', 'Note_Type', 'Note_Reason', 'Provider_Anomaly_Score', 'Category_Rank', 'Narrate_Selected',
#          'Patient_Mix_Share']
# schema = StructType([StructField(c, StringType(), True) for c in _cols])
# _out = client_df[_cols].astype(str)        # client rows only: quarantined narratives never reach the table
# if not _out.empty:
#     df_spark = spark.createDataFrame(_out, schema=schema)
#     if not spark.catalog.tableExists(test_table_llm_summarization):
#         df_spark.write.mode('overwrite').saveAsTable(test_table_llm_summarization)
#     else:
#         df_spark.write.mode('append').option('mergeSchema', 'true').saveAsTable(test_table_llm_summarization)

##### Trust report

In [ ]:
# ── Trust report: read before sharing the notes ──
print("=" * 72)
print(f"Run {NOTEBOOK_VERSION} | dictionary {DICTIONARY_VERSION} | scoring date {scoring_date} | referrals: {REFERRAL_SOURCE}")
print(f"Referred provider-categories: {len(providers)} | in importance file but not referred (skipped): "
      f"{load_stats['importance_only_not_referred']} | referred without feature rows: {load_stats['referred_without_feature_rows']}")
print("Note types:", output_df['Note_Type'].value_counts().to_dict())
print("QA status:", output_df['QA_Status'].value_counts().to_dict())
_narr = output_df[output_df['QA_Status'].isin(['pass', 'needs_review'])]
if len(_narr):
    print(f"  narratives passing QA: {(_narr['QA_Status'] == 'pass').mean():.0%} of {len(_narr)}")
    _fc = Counter(f.split(':')[0] + (':' + f.split(':')[1].split('(')[0] if f.startswith('banned:') else '')
                  for flags in _narr['QA_Flags'] for f in flags.split(', ') if f and f != 'pass')
    print("  QA flags:", dict(_fc.most_common()))
    print("  reasoning levels:", _narr['Reasoning_Level'].value_counts().to_dict())
    _l2n = lambda c: int((_narr[c].astype(str).str.len() > 0).sum())
    print(f"  Layer 2 used: {int(_narr['Layer2_Used'].sum())}; candidates dropped: {_l2n('Layer2_Dropped')}; "
          f"linked schemes proposed for review (Layer2_Added): {_l2n('Layer2_Added')}; menu schemes without a dictionary link "
          f"(Layer2_Suggested): {_l2n('Layer2_Suggested')}; invented names rejected: {_l2n('Layer2_Rejected')}; "
          f"unusable answers: {_l2n('Layer2_Note')}")
print(f"Feature roles: {feat['Role'].value_counts().to_dict()}")
print(f"  reasons: {feat.loc[feat['Role_Reason'] != '', 'Role_Reason'].value_counts().to_dict()}")
_shown = feat[feat['Shown_As'].isin(['Main', 'Supporting'])]
if len(_shown):
    print(f"  patterns shown to the LLM with a Python-computed value: {(_shown['Value_Sentence'] != '').mean():.0%}")
    print(f"  main pattern stability: {feat.loc[feat['Shown_As'] == 'Main', 'Stability'].value_counts().to_dict()}")
    print(f"  main pattern consensus: {feat.loc[feat['Shown_As'] == 'Main', 'Consensus'].value_counts().to_dict()}")
print("  value sources:", dict(value_coverage))
if len(feat[feat['Shown_As'] == 'Main']):
    print("  main patterns by Definition_Confidence:", feat.loc[feat['Shown_As'] == 'Main', 'Definition_Confidence'].value_counts().to_dict())
if 'ENC_CLM_IND' in df_scoring_master.columns:
    print("  ENC_CLM_IND by category (check how PMT_AMT is filled for encounters):")
    print(df_scoring_master.groupby('Category')['ENC_CLM_IND'].value_counts(dropna=False).unstack(fill_value=0).to_string())
print(f"Patient mix providers (model review list): {int((providers['Note_Type'] == 'patient_mix').sum())} "
      f"{providers.loc[providers['Note_Type'] == 'patient_mix', 'Category'].value_counts().to_dict()}")
print("Scheme strength (deterministic):", providers['Scheme_Strength'].replace('', 'none').value_counts().to_dict())
print("Detected schemes (narratives):", output_df.loc[output_df['Note_Type'] == 'narrative', 'Detected_Schemes'].value_counts().head(10).to_dict())
if len(audit_df):
    _retry = int((audit_df['Attempts'] > 1).sum())
    print(f"Generations needing a retry (at {RETRY_REASONING} effort): {_retry} of {len(audit_df)}; failed: "
          f"{int((output_df['QA_Status'] == 'generation_failed').sum())}; hit the token limit on the first attempt: "
          f"{int(audit_df['First_Attempt_Hit_Limit'].astype(bool).sum())}; median new tokens: {audit_df['New_Tokens'].median():.0f}")
    if len(audit_df) >= 10 and _retry / len(audit_df) > 0.3:
        print(f"  More than 30% of narratives needed the retry: set NARRATIVE_REASONING['default'] = 'low' (budget {REASONING_TOKEN_BUDGET['low']}).")
if _unknown_tiers:
    print(f"Importance_Tier values outside {sorted(MAIN_TIERS)} (never main patterns): {_unknown_tiers}")
if UNKNOWN_LEVELS:
    print(f"Untranslated tier labels (shown as 'an unusual level'): {sorted(UNKNOWN_LEVELS)[:10]}")
_seen = set(_shown['Feature']) if len(_shown) else set()
_no_desc = sorted({c for c in CODES_WITHOUT_DESCRIPTION if any(c in f for f in _seen)})
_grp = sorted({c for c in CODES_DESCRIBED_BY_GROUP if any(c in f for f in _seen)})
print(f"Codes shown with only their ICD-10 group: {len(_grp)} {_grp[:5]} | with no description: {len(_no_desc)} {_no_desc[:8]}")
if brief_scrub_hits:
    print(f"Dictionary wording scrubbed before the LLM saw it (fix in the dictionary): {dict(brief_scrub_hits)}")
print(f"Client file: {len(client_df)} rows (pass, evidence_only, patient_mix, evidence_fallback); quarantine: {len(quarantine_df)} "
      "(needs_review, generation_failed). Work the quarantine file first; its providers already have an evidence note in the client file.")
print("=" * 72)